In [1]:
def get_OLD_sub():
    
    import numpy as np
    import pandas as pd
    import pickle 
    import warnings
    import os
    import re
    from scipy.stats import kurtosis, zscore, skew, iqr
    warnings.filterwarnings("ignore")
    
    import tensorflow.keras.backend as K
    import tensorflow.keras.layers as layers
    from tensorflow.keras.callbacks import Callback, ReduceLROnPlateau, ModelCheckpoint, EarlyStopping
    import tensorflow as tf
    
    
    def create_mlp1_g(num_columns, num_labels, hidden_units, dropout_rates, learning_rate):
    
        inp = tf.keras.layers.Input(shape = (num_columns, ))
        inp = tf.keras.layers.GaussianNoise(.03, seed = 42)(inp)
        xinp = inp
        
        
        x = tf.keras.layers.BatchNormalization()(inp)
    
        #att1 = tf.keras.layers.Reshape((5, 67))(x)
    
        #attn = tf.keras.layers.Attention(use_scale = True)([att1, att1])
        #attn = tf.keras.layers.Flatten()(attn)
        #x = tf.keras.layers.Concatenate()([x, attn])
    
        x = tf.keras.layers.Dropout(dropout_rates[0])(x)
        for i in range(len(hidden_units)): 
            x = tf.keras.layers.Dense(hidden_units[i])(x)
            x = tf.keras.layers.BatchNormalization()(x)
            x = tf.keras.layers.Activation("swish")(x)
            x = tf.keras.layers.Dropout(dropout_rates[i+1])(x)   
            x = tf.keras.layers.Concatenate()([x, xinp])
            xinp = x
    
        x = tf.keras.layers.Dense(512)(x)
        x = tf.keras.layers.Activation("swish")(x)
        x = tf.keras.layers.Dense(64)(x)
    
        x = tf.keras.layers.Activation("swish")(x)
    
        
        out = tf.keras.layers.Dense(num_labels)(x)
        #out = tf.keras.layers.Activation('linear')(x)
        
        model = tf.keras.models.Model(inputs = inp, outputs = out)
        model.compile(optimizer = tf.keras.optimizers.Adam(learning_rate = learning_rate),
                    loss = tf.keras.losses.MeanAbsoluteError() ,
                          #steps_per_execution = 32
                          #metrics = ["mae"], 
                         )
            
        return model
    
    
    def create_mlp1(num_columns, num_labels, hidden_units, dropout_rates, learning_rate):
    
        inp = tf.keras.layers.Input(shape = (num_columns, ))
        xinp = inp
        
        
        x = tf.keras.layers.BatchNormalization()(inp)
    
        #att1 = tf.keras.layers.Reshape((5, 67))(x)
    
        #attn = tf.keras.layers.Attention(use_scale = True)([att1, att1])
        #attn = tf.keras.layers.Flatten()(attn)
        #x = tf.keras.layers.Concatenate()([x, attn])
    
        x = tf.keras.layers.Dropout(dropout_rates[0])(x)
        for i in range(len(hidden_units)): 
            x = tf.keras.layers.Dense(hidden_units[i])(x)
            x = tf.keras.layers.BatchNormalization()(x)
            x = tf.keras.layers.Activation("swish")(x)
            x = tf.keras.layers.Dropout(dropout_rates[i+1])(x)   
            x = tf.keras.layers.Concatenate()([x, xinp])
            xinp = x
    
        x = tf.keras.layers.Dense(512)(x)
        x = tf.keras.layers.Activation("swish")(x)
        x = tf.keras.layers.Dense(64)(x)
    
        x = tf.keras.layers.Activation("swish")(x)
    
        
        out = tf.keras.layers.Dense(num_labels)(x)
        #out = tf.keras.layers.Activation('linear')(x)
        
        model = tf.keras.models.Model(inputs = inp, outputs = out)
        model.compile(optimizer = tf.keras.optimizers.Adam(learning_rate = learning_rate),
                    loss = tf.keras.losses.MeanAbsoluteError() ,
                          #steps_per_execution = 32
                          #metrics = ["mae"], 
                         )
            
        return model
    
    
    
    sc_dictV3 = pickle.load(open("/kaggle/input/rohlik-v43/sc_dictV3.pkl", "rb"))
    sc_dictV4 = pickle.load(open("/kaggle/input/rohlik-v50/MLPV4_ss_dict.pkl", "rb"))
    sc_dict = pickle.load(open("/kaggle/input/mlpv2-misc/new_ss_dict.pkl", "rb"))
    
    fruits = {'dinosaur eggs', 'star apple', 'nopal fruit', 'sunset apple', 'date plums', 'egg fruit', 'maracuya', 'lancetilla mango', 'red delicious apple', 'oeillade noire grape', 'marula', 'green raisins', 'umeboshi', 'limquat', 'quararibea', 'indian persimmon', 'cosmic crisp apple ', 'wild sugar apple', 'ambrosia melon', 'red mombin', 'valencia oranges', 'black sapote', 'pink pomelo', 'mangaba', 'damson plums', 'meyer lemon', 'rollinia', 'zill mango', 'alpine strawberry', 'limau bali', 'granny smith apple', 'pinova apples', 'batuan ', 'splendor apple', 'midyim berry', 'highbush cranberries', 'orient melon', 'zestar apple', 'xoconostle cactus fruit ', 'tachibana orange', 'african breadfruit', 'honeysuckles', 'fazli mango', 'northern spy apples', 'horned melon', 'red pear', 'zig zag vine fruit', 'jamun plum', 'green banana', 'fairchild tangerines', 'bilberries ', 'plums', 'annona', 'hala fruit', 'barbados cherries', 'zinfandel grape', 'durian', 'passion fruit', 'oriental cherry', 'ortanique', 'marang', 'kakadu plum', 'cabernet sauvignon grape ', 'rough lemon', 'wax gourd', 'wampee', 'winter squash', 'keitt mango', 'o’henry peach', 'boysenberry', 'pink lady apples', 'persian limes', 'ogallala strawberry', 'orange', 'osteen mango', 'desert fig', 'gooseberry', 'hog plums', 'ximenia caffra ', 'ichigo', 'acerola cherries', 'wild water lemon', 'blue watermelon', 'kutjera fruit', 'rangpur lime', 'rambai', 'pequi fruit', 'caper berry', 'honeyberries', 'rose apple', 'juicy tomatoes', 'brazilian guava ', 'nectacotum', 'parsonage pears', 'himalayan mulberries', 'eggplant', 'papaya', 'quandong', 'korean pear', 'lady apple', 'zarzamora', 'wild strawberry', 'cedar bay cherry ', 'zalzalak', 'ogeechee lime', 'youngberry', 'bottle gourd', 'cranberry', 'blood orange', 'fibrous satinash fruit', 'okuzgozu grape', 'pacific rose apple', 'batuan', 'courgette', 'west indian cherry', 'jazz apples', 'sudachi', 'lippens mango', 'langsat', 'macoun apple', 'grapefruit', 'avocado', 'nungu fruit', 'jackfruit', 'huckleberry', 'usuma fruit', 'queen’s forelle pear', 'kasturi mango', 'orient pear', 'pumpkins', 'jostaberry', 'vaccarese grapes', 'petit rouge grapes', 'hyuganatsu', 'otaheite gooseberry', 'akebi fruit', 'topaz apple', 'oso grande strawberry', 'mayan nut', 'canistel fruit', 'zweigelt grape', 'juniper berries', 'ensete', 'orin apple', 'tourist pineapple', 'elephant apple (chalta)', 'dragon fruit', 'indian prune', 'le conte pear', 'ground plum', 'coco plum ', 'green lemon', 'evergreen huckleberry', 'miracle fruit', 'satsuma mandarin', 'coconut', 'hackberry', 'fascell mango', 'ziziphus mauritiana', 'ya pear', 'mcintosh apple', 'pink pineapple', 'yellow guava ', 'husk tomatoes', 'nepali hog plums', 'may pride peaches', 'red bayberry', 'red passion fruit', 'black currant', 'urava fruit ', 'yayat palm', 'blue java banana ', 'limeberry', 'asam kumbang', 'peach palm fruit', 'lemon aspen', 'sugar plum', 'galia melon', 'xigua fruit ', 'marsh pink grapefruit', 'ambrosia apples', 'olive', 'yellow dragon fruit', 'nam dok mai', 'nance', 'opuntia', 'desert quandong', 'japanese persimmon', 'amla', 'rosigold mango', 'java apple', 'louise bonne of jersey pear', 'tommy atkins mango', 'ita palm fruit', 'lima orange', 'lemato', 'madison peach', 'victoria plum', 'farkleberry', 'newton pippin apple', 'pomegranate', 'forest strawberries', 'red banana', 'green bell pepper', 'indian jujube fruit', 'mango', 'annatto', 'queen apple', 'pineapple', 'cortland apple ', 'sugar berry', 'darwin’s barberries', 'betel nut', 'white currant', 'five flavor berry', 'langra mango', 'camu camu berry', 'pinot noir grapes', 'ozark gold apple', 'cleopatra mandarin', 'lime', 'mexican limes ', 'hawthorn fruit', 'etrog', 'olallieberry', 'governor’s plum', 'quince ', 'lilly pilly', 'xarel-lo ', 'umbu fruit', 'york apple', 'indian almond fruit', 'yellow strawberry', 'emblica', 'pie pumpkins', 'plumcots', 'honey crisp apples', 'dabai fruit', 'entawak', 'oval kumquat', 'blue dragon fruit', 'raspberry', 'imbu ', 'nectarines', 'imbe', 'wild peach', 'green fig', 'rockmelon', 'kvede', 'johannisbeere', 'ozark beauty strawberry', 'false mastic fruit', 'zabergau reinette apple', 'yellow chili pepper', 'velvet tamarind', 'cherry', 'pili fruit', 'yellow plum', 'riberry', 'indonesian lime', 'mandarin', 'green adjou pear', 'taylor’s gold pear', 'uvilla fruit', 'marionberry', 'yayat', 'cornelian cherry', 'may apple', 'rose hip', 'voavanga fruit', 'fox grapes', 'jelly palm fruit', 'oroblanco', 'kaffir lime', 'wahoo', 'ximenia americana ', 'calabash', 'iboga', 'white mulberries', 'nuts', 'zawngtah', 'strawberry guava ', 'red mulberry', 'ugli fruit', 'fuji apples', 'williams pear', 'highbush blueberries', 'yellow sapote', 'black cherries', 'yucca', 'zwetschge', 'texas persimmon', 'kei apple.', 'green tomato', 'tomatillo', 'raspuri mango', 'blueberries', 'nagami kumquat', 'breadfruit ', 'tart cherry', 'lapsi', 'walnut', 'discovery apple', 'rumdul', 'maqui', 'van dyke mango', 'nashi pears', 'churchland pear', 'peaches', 'canary fruit', 'jatoba t', 'junglesop fruit', 'icacina', 'pineberries', 'monster deliciosa', 'brazil nut', 'cantaloupe', 'green grapes', 'pluots', 'citron', 'emu apple fruit', 'beechnut', 'european pear', 'yumberry', 'oullins gage plum', 'carambola', 'enterprise apple', 'madrono', 'figs', 'amanatsu oranges', 'chinese quince ', 'shonan gold ', 'merlot grape', 'naranjilla', 'long neck avocado', 'pigeon plum', 'orangelo', 'kanzi apple', 'green egg squash ', 'mammee apple', 'yemenite citron', 'strawberry', 'minneolo tangelo', 'early girl tomato', 'rajka apples', 'green gage plum', 'valencia pride mango', 'villafranca lemon', 'water apple', 'redcurrant', 'blood lime', 'mexican lime', 'volkamer lemon ', 'vanilla', 'early gold mango', 'white-flowered gourd', 'kiwano', 'apricots', 'terap', 'wild banana', 'double coconut', 'bacuri fruit', 'green strawberry', 'loquat', 'beach plums', 'nageia', 'finger lime', 'blue pearmain apple', 'ilama', 'chinese white pear ', 'ugni berry', 'western hackberry', 'tropical apricot', 'asian pears', 'xinomavro ', 'lemon', 'trifoliate orange', 'black raspberries', 'kyoho grape', 'gamboge', 'yangmei', 'pomato', 'dangle berry', 'indian sherbet berry', 'lady finger banana', 'sour cherry', 'ice cream bean fruit', 'lemon drop melon', 'kaywa', 'red huckleberry', 'umbra fruit', 'lambkin melon', 'kwai muk', 'paw paw', 'fukushu kumquat', 'opal plum', 'wax apple', 'quinault strawberry', 'honey locust', 'persimmons', 'red grape', 'kawakawa', 'golden apple', 'african mango', 'yali pear', 'prickly pears', 'sweet lime', 'carob ', 'yellow watermelon', 'bell peppers ', 'giant lau lau', 'genip', 'marisol clementine', 'jujube fruit', 'chinoto sour orange ', 'karonda berry', 'dracontomelon', 'melon pear fruit', 'salak ', 'kabosu', 'heirloom tomatoes', 'querina apples', 'arava melon', 'clementine', 'yuzu', 'feijoa', 'xerophyte', 'guavaberry', 'biriba', 'wild bacher grape', 'wild lowbush blueberry', 'bearberries', 'orlando tangelo', 'nutmeg', 'crab apple', 'fe’i banana', 'nere fruit', 'sweet granadilla', 'pink pearl apples', 'dekopon', 'tamarillo', 'mamey sapote', 'pitaya', 'wealthy apple', 'gorham pear', 'young mango', 'darling plum', 'zhe', 'margil apples', 'duku fruit', 'blue olive', 'cavendish banana ', 'indian gooseberry', 'pigface fruit', 'crimson gold apple', 'key lime', 'mock strawberry', 'neem', 'golden delicious apple', 'madrono ', 'grape', 'black mulberry ', 'pears', 'queen anne cherry', 'dates', 'oregon grape', 'knobby russet apple', 'watermelon', 'wild orange', 'hazelnuts', 'ice apple', 'cempedak ', 'star fruit', 'huito', 'tangerine', 'wild mangosteen', 'hawaiian mountain apples', 'tompkins king apple', 'mamoncillo', 'flatwoods plum', 'ambarella', 'wolf berry', 'loganberry', 'granadilla', 'peumo fruit', 'balsam apples ', 'yellow apple', 'naval oranges', 'plantains', 'grosella negra', 'queen tahiti pineapple', 'opal apple', 'crimson delight apple', 'guarana.', 'florida cherry', 'honeydew melon', 'bitter melon', 'rhobs el arsa', 'japanese plum', 'golden kiwi', 'blackberries ', 'mangosteen', 'conference pear', 'narenj', 'hardy kiwi', 'vespolina grape', 'lucuma', 'sloe', 'yellow bell peppers', 'kiwi ', 'zebra melon', 'tangelo', 'lodi apple', 'greek fig', 'gala apple', 'maypop', 'naartjie', 'blue sausage fruit', 'kumquats ', 'lord lambourne apple', 'guava', 'desert lime', 'pomelo', 'tayberry', 'custard apple', 'sand cherry', 'okra', 'cucumber', 'grumichama', 'dewberries', 'illawarra plum', 'natal plum', 'black apples', 'rambutan', 'xylocarp ', 'goji berry', 'wild lemon', 'roselle', 'elands sour fig', 'vicar of winkfield pear', 'lingonberry', 'raisins', 'nectacot', 'lablab fruit', 'sugar baby watermelon ', 'velvet apple', 'ponderosa lemons', 'oil palm', 'prunes', 'rocha pear', 'rata', 'dodder laurel', 'soncoya', 'emu berry fruit', 'xanthium strumarium ', 'sour plum ', 'crispin apple', 'rumbus parviflorus', 'kaki persimmon', 'ramontchi', 'barberries ', 'goumi', 'babaco', 'yellow passion fruit ', 'dessert banana', 'macadamia nut', 'brush cherry', 'green apple', 'blue marble fruit', 'pecan', 'elderberry', 'incaberry', 'zante currant', 'santol', 'ububese ', 'davidson’s plums', 'thimberry', 'korlan', 'water chestnut', 'dead man’s finger', 'winter melon', 'grand nain banana', 'nocera grape', 'serviceberry ', 'white aspen', 'amaou strawberries', 'soursop', 'liberty apple', 'zierfandler grape', 'kahikatea', 'eastern hawthorn fruit', 'palestinian sweet limes', 'yunnan hackberry', 'saigon mango.', 'kowai', 'icaco'}
    
    vegetables = {'shallot', 'bean sprout', 'cheddar cauliflower', 'molokhia', 'udupi mattu gulla eggplant.', 'collard greens', 'leek', 'cherry pepper', 'eddoe', 'fava bean', 'zephyr squash ', 'purple carrot', 'mushroom ', 'lima bean ', 'butternut squash', 'ulluco', 'xanthosoma brasiliense', 'fluted pumpkin', 'ash gourd', 'flint corn', 'chamomile', 'vidalia onion', 'yam', 'all blue potato', 'new zealand spinach', 'broccoflower ', 'radish', 'gigante bean', 'nettle', 'russian banana potato', 'stokes purple sweet potato', 'garlic', 'celery  ', 'arugula', 'lavender', 'black radish', 'taro', 'hot pepper', 'bitter melon', 'chickpeas', 'soybean', 'hubbard squash', 'radicchio', 'navy bean', 'eggplant ', 'flat beans', 'topinambur', 'gold rush squash', 'snap pea', 'xanthosoma sagittifolium', 'great northern bean', 'savoy cabbage', 'kohlrabi', 'mallow', 'onion', 'sweet corn', 'chives', 'yellow onion', 'white bell pepper ', 'daikon', 'red onion', 'carnival squash', 'purple lettuce', 'boniato', 'tat soi', 'french sorrel', 'tabasco pepper', 'red carrot', 'red kuri squash', 'velvet bean', 'corn', 'dill', 'yellow summer squash', 'anise', 'red wing onion', 'spinach', 'belgian endive', 'corms', 'yukon gold potatoes', 'salsify ', 'dickinson pumpkin', 'white asparagus', 'red cabbage', 'arame', 'upland cress', 'cannellini bean', 'chili pepper', 'purple kale', 'bamboo shoot', 'pinto bean', 'red gold potato', 'white 7 pot pepper', 'azuki bean', 'yukon gold potato', 'french bean', 'honeynut squash', 'coriander', 'hubbard squash.', 'purple tomato', 'okra', 'spaghetti squash', 'lentil', 'cucumber', 'xà lách', 'delicata', 'new potato', 'jerusalem artichoke', 'flat italian onion', 'red bell pepper ', 'torpedo onion', 'peruvian purple potato', 'flat white boer pumpkin', 'artichoke', 'patty pan squash.', 'pea', 'jicama', 'asian greens', 'mung bean', 'orange tomato', 'split pea', 'sweet potato', 'cauliflower', 'german butterball potato', 'scallion ', 'futsu squash', 'kale', 'field cucumber', 'parsnip', 'fat hen', 'pepper', 'baby boo pumpkin', 'bell pepper', 'fiddlehead', 'chard ', 'kidney bean', 'malanga', 'squash', 'skirret', 'carrot', 'purple majesty potato', 'cabbage', 'red bliss potato', 'amaranth', 'vivaldi potato', 'mangelwurzel', 'purple bell pepper ', 'black bean', 'ambercup squash', 'cayenne pepper', 'oregano', 'gem squash', 'lettuce', 'broad-leaf arrowhead', 'arrowroot', 'mustard greens', 'adzuki bean', 'white corn', 'basil', 'wax bean', 'frisée', 'lumina pumpkin', 'arikara squash ', 'marjoram', 'peanut', 'purple radish', 'red warty thing squash', 'purple brussels sprout', 'broad beans', 'fiorentino tomato', 'yellow tomato', 'banana squash', 'red russian kale', 'bintje potato', 'alfalfa sprouts', 'orange bell pepper', 'purple corn', 'celeriac', 'purple cabbage', 'rhubarb', 'avocado', 'black-eyed pea', 'paprika', 'vitelotte potato', 'konjac', 'japanese sweet potato', 'ginger', 'oak leaf lettuce ', 'wasabi', 'morels', 'broccoli ', 'pie pumpkin', 'borage', 'rosemary', 'borlotti bean', 'xi yang ca', 'runner bean', 'green bean', 'water chestnut', 'zucchini ', 'ube', 'mangetout', 'tomato', 'brussels sprout', 'broccolini ', 'cymbopogon', 'purple cauliflower', 'turnip', 'endive', 'watercress', 'rutabaga', 'purple asparagus', 'acorn squash', 'caraway', 'tomatillo', 'ahipa', 'bok choy', 'kelp', 'black salsify', 'beet', 'asparagus', 'urad bean', 'honey bear squash ', 'aonori', 'habanero pepper', 'xi lan hua', 'pattypan squash', 'butterhead lettuce', 'potato', 'fennel', 'boston morrow squash', 'miner’s lettuce', 'jalapeño', 'fayot bean', 'purple string bean', 'burdock root', 'butter bean', 'thyme', 'banana pepper'}
    
    path = "/kaggle/input/rohlik-sales-forecasting-challenge-v2/"
    
    vers = 2
    # model path
    mpath = f"/kaggle/input/rohlik-v{vers}/v{vers}/"
    
    ord_enc = pickle.load(open(mpath + "ord_enc.pkl", "rb"))
    sales_dict = pickle.load(open(mpath + "sales_dict.pkl", "rb"))
    sell_price_dict = pickle.load(open(mpath + "sell_price_dict.pkl", "rb"))
    #cor_id_dict = pickle.load(open(mpath + "cor_id_dict.pkl", "rb"))
    cor_val_dict = pickle.load(open(mpath + "cor_val_dict.pkl", "rb"))
    
    cor_mean_dict = pickle.load(open("/kaggle/input/cor-mean-dict/cor_mean_dict.pkl", "rb"))
    cor_median_dict = pickle.load(open("/kaggle/input/cor-median-dict/cor_median_dict.pkl", "rb"))
    cor_range_dict = pickle.load(open("/kaggle/input/cor-range-dict/cor_range_dict.pkl", "rb"))
    cor_std_dict = pickle.load(open("/kaggle/input/cor-std-dict/cor_std_dict.pkl", "rb"))
    cor_id_dict_pos = pickle.load(open("/kaggle/input/cor-id-dict/cor_id_dict_pos.pkl", "rb"))
    cor_id_dict_neg = pickle.load(open("/kaggle/input/cor-id-dict/cor_id_dict_neg.pkl", "rb"))
    
    
    week_day_sales_dict = pickle.load(open("/kaggle/input/new-pipe-dicts/week_day_sales_dict.pkl", "rb"))
    week_day_avail_dict = pickle.load(open("/kaggle/input/new-pipe-dicts/week_day_avail_dict.pkl", "rb"))
    
    dpath = "/kaggle/input/rohlik-v28/"
    single_order_z_dict = pickle.load(open(dpath + "single_order_z_dict/single_order_z_dict.pkl", "rb"))
    tot_kurt_dict = pickle.load(open(dpath + "tot_kurt_dict/tot_kurt_dict.pkl", "rb"))
    tot_orders_z_dict = pickle.load(open(dpath + "tot_orders_z_dict/tot_orders_z_dict.pkl", "rb"))
    tot_skew_dict = pickle.load(open(dpath + "tot_skew_dict/tot_skew_dict.pkl", "rb"))
    
    train = pd.read_csv(path + "sales_train.csv")
    test = pd.read_csv(path + "sales_test.csv")
    calendar = pd.read_csv(path + "calendar.csv")
    inventory = pd.read_csv(path + "inventory.csv")
    
    
    warehouse_map = {
        'Budapest_1': 0, 'Prague_2': 2,
        'Brno_1': 4, 'Prague_1': 1, 'Prague_3': 3,
         'Munich_1': 5, 'Frankfurt_1': 6
         }
    
    train = pd.read_csv(path + "sales_train.csv")
    
    
    train.sort_values(by = ["warehouse", "unique_id", "date"], inplace = True)
    train.reset_index(drop = True, inplace = True)
    
    final_preds = pd.DataFrame(columns = ["id", "sales"])
    
    
        
    # DAT
    train["date"] = pd.DatetimeIndex(train["date"])
    
    train["day"] = train["date"].dt.day
    train["week"] = train["date"].dt.dayofweek
    train["week_of_year"] = train["date"].dt.isocalendar().week
    train["month"] = train["date"].dt.month
    train["quarter"] = train["date"].dt.quarter
    train["year"] = train["date"].dt.year - 2020
    train["day_of_year"] = train["date"].dt.dayofyear
    train["day_num_from_start"] = (train["year"]  * 366) + train["day_of_year"]
    train["week_num_from_start"] = (train["year"]  * 52) + train["week_of_year"]
    train["month_num_from_start"] = (train["year"]  * 12) + train["month"]
    train["quarter_num_from_start"] = (train["year"]  * 4) + train["quarter"]
    train["week_of_month"] = train["day"] // 7
    
    ###############################################################################
    
    ## when train is sorted
    rnk = train.groupby(["warehouse", "unique_id"])["day_num_from_start"].transform("rank", ascending = False)
    
    ## need last 30 days records, since I am using last 30 days aggregations
    train = train.loc[rnk<=30]
    train["rank"] = train.groupby(["warehouse", "unique_id"])["day_num_from_start"].transform("rank")
    
    ###############################################################################
    
    # DATE
    test["date"] = pd.DatetimeIndex(test["date"])
    
    test["day"] = test["date"].dt.day
    test["week"] = test["date"].dt.dayofweek
    test["week_of_year"] = test["date"].dt.isocalendar().week
    test["month"] = test["date"].dt.month
    test["quarter"] = test["date"].dt.quarter
    test["year"] = test["date"].dt.year - 2020
    test["day_of_year"] = test["date"].dt.dayofyear
    test["day_num_from_start"] = (test["year"]  * 366) + test["day_of_year"]
    test["week_num_from_start"] = (test["year"]  * 52) + test["week_of_year"]
    test["month_num_from_start"] = (test["year"]  * 12) + test["month"]
    test["quarter_num_from_start"] = (test["year"]  * 4) + test["quarter"]
    test["week_of_month"] = test["day"] // 7
    ###############################################################################
    
    train["sample"] = "train"
    test["sample"] = "test"
    
    ###############################################################################
    
    test["date_num"] = test["date"].transform("rank", method = "dense")
    
    ###############################################################################
    
    ## This loop will fetch us predictions starting from 2024-06-03 to 2024-06-16
    
    for cur_day in range(1, 15):
    
        ###############################################################################
        
    
        
        cur = test[test["date_num"]==cur_day].copy()
        
        cur = pd.concat((train, cur), axis = 0)
        
        cur.sort_values(by = ["warehouse", "unique_id", "date"], inplace = True)
        cur.reset_index(drop = True, inplace = True)
        
        cur_drop = ["sample", "date_num"]
        
        
        ###############################################################################
        
        cur = pd.merge(cur, inventory, on = "unique_id", how = "left")
        cur.drop(columns = "warehouse_y", inplace = True)
        cur.rename(columns = {"warehouse_x": "warehouse"}, inplace = True)
        
        #cur["date"] = pd.DatetimeIndex(cur["date"])
        calendar["date"] = pd.DatetimeIndex(calendar["date"])
        
        # idx_to_wh_map = {v:k for k,v in warehouse_map.items()}
        
        # cur["warehouse"] = cur["warehouse"].map(idx_to_wh_map)
        
        cur = pd.merge(cur, calendar, on = ["date", "warehouse"] , how = "left")
        
        cur["warehouse"] = cur["warehouse"].map(warehouse_map)
        
        
        names = cur["name"].str.split("_", expand = True)
        
        def return_fruit(x):
            if x.lower() in fruits:
                return x 
            return np.nan
        
        def return_veggy(x):
            if x.lower() in vegetables:
                return x 
            return np.nan
        
        
        cur["fruit"] = names[0].apply(return_fruit)
        
        
        cur["veggy"] = names[0].apply(return_veggy)
        
        ##############################################################################
    
        # this is not exactly single order value, as  
        # it is total orders placed for all products for that day for that warehouse 
    
        cur["single_order_value"] = cur["total_orders"] / cur["sell_price_main"]
    
        cur["single_order_value"] = cur["single_order_value"].astype("float32")
    
        ###############################################################################
        
        ###############################################################################
        
        to_drop = ["holiday_name", "date", 
                   "name", "L1_category_name_en", "L2_category_name_en", "L3_category_name_en",
                   "L4_category_name_en",  "fruit", "veggy", "availability"]
        
        ###############################################################################
        
        warehouse_map = {
            'Budapest_1': 0, 'Prague_2': 2,
            'Brno_1': 4, 'Prague_1': 1, 'Prague_3': 3,
             'Munich_1': 5, 'Frankfurt_1': 6
             }
        
        #cur["warehouse"] = cur["warehouse"].map(warehouse_map)
        
        ###############################################################################
        
        # L 
        
        Ls = ["L2_category_name_en", "L3_category_name_en", "L4_category_name_en"]
        L_df = {}
        L_sum = 0
        
        for col in Ls:
             L_df[col + "grp3"] = cur[col].str.split("_", expand = True)[2].astype("int16")
             L_sum += L_df[col + "grp3"].astype("int16")
        
        L_df["L_sum"] = L_sum 
        del L_sum
        
        ###############################################################################
        
        
        # Ordinal Encoder
        
        ord_cols = ["name", "fruit", "veggy", "L2_category_name_en", "L3_category_name_en", "L4_category_name_en"]
        
        ord_df = ord_enc.transform(cur[ord_cols])
        
        ord_df = pd.DataFrame(ord_df, columns = [i + "_enc" for i in ord_cols])
        
        ###############################################################################
    
        for day in [3,7,14,30]:
    
    
            mn = cur.groupby(["warehouse","unique_id"])["sales"].shift(1).rolling(window = day).agg("mean").astype("float32")
            for k,v in zip((cur["unique_id"].astype(str) + "_" + cur["date"].astype(str)), mn):
                if (str(day) + "_" + k) not in cor_mean_dict:
                    cor_mean_dict[str(day) + "_" + k] = v
                
                
                
            med = cur.groupby(["warehouse","unique_id"])["sales"].shift(1).rolling(window = day).agg("median").astype("float32")
            for k,v in zip((cur["unique_id"].astype(str) + "_" + cur["date"].astype(str)), med):
                if (str(day) + "_" + k) not in cor_median_dict:
                    cor_median_dict[str(day) + "_" + k] = v
                
                
            
            std = cur.groupby(["warehouse","unique_id"])["sales"].shift(1).rolling(window = day).agg("std").astype("float32")
            for k,v in zip((cur["unique_id"].astype(str) + "_" + cur["date"].astype(str)), std):
                if (str(day) + "_" + k) not in cor_std_dict:
                    cor_std_dict[str(day) + "_" + k] = v
                
           
            
            Max = cur.groupby(["warehouse","unique_id"])["sales"].shift(1).rolling(window = day).agg("max").astype("float32")
            Min = cur.groupby(["warehouse","unique_id"])["sales"].shift(1).rolling(window = day).agg("min").astype("float32")
            ran = (Max - Min).astype("float32")
            for k,v in zip((cur["unique_id"].astype(str) + "_" + cur["date"].astype(str)), ran):
                if (str(day) + "_" + k) not in cor_range_dict:
                    cor_range_dict[str(day) + "_" + k] = v
        
        ###############################################################################
        ###############################################################################
        
    
        cur["1"] = cur["unique_id"].apply(lambda x: cor_id_dict_pos[x][0] if cor_id_dict_pos[x] else np.nan).apply(lambda x: f"{x:.0f}")
        cur["2"] = cur["unique_id"].apply(lambda x: cor_id_dict_pos[x][1] if cor_id_dict_pos[x] else np.nan).apply(lambda x: f"{x:.0f}")
        cur["3"] = cur["unique_id"].apply(lambda x: cor_id_dict_pos[x][2] if cor_id_dict_pos[x] else np.nan).apply(lambda x: f"{x:.0f}")
        cur["4"] = cur["unique_id"].apply(lambda x: cor_id_dict_pos[x][3] if cor_id_dict_pos[x] else np.nan).apply(lambda x: f"{x:.0f}")
        
        
        cur["11"] = cur["unique_id"].apply(lambda x: cor_id_dict_neg[x][0] if cor_id_dict_neg[x] else np.nan).apply(lambda x: f"{x:.0f}")
        cur["22"] = cur["unique_id"].apply(lambda x: cor_id_dict_neg[x][1] if cor_id_dict_neg[x] else np.nan).apply(lambda x: f"{x:.0f}")
        cur["33"] = cur["unique_id"].apply(lambda x: cor_id_dict_neg[x][2] if cor_id_dict_neg[x] else np.nan).apply(lambda x: f"{x:.0f}")
        cur["44"] = cur["unique_id"].apply(lambda x: cor_id_dict_neg[x][3] if cor_id_dict_neg[x] else np.nan).apply(lambda x: f"{x:.0f}")
        
        
        # prev days sales for correlated unique_id's
        # cur["1"] = cur["1"] + "_" + (cur["date"] - pd.DateOffset(1)).astype(str)
        
        
        for i in range(1, 5):
            
            for day in [1,3,7,14,30]:
                
                
                
                cur[f"{i}_POS_corr_prev_{day}"] = ((cur[str(i)] + "_" + \
                    (cur["date"] - pd.DateOffset(day)).astype(str)).apply(lambda x: cor_val_dict.get(x, np.nan))).astype("float32")
                
                cur[f"{i}_NEG_corr_prev_{day}"] = ((cur[str(i) + str(i)] + "_" + \
                    (cur["date"] - pd.DateOffset(day)).astype(str)).apply(lambda x: cor_val_dict.get(x, np.nan))).astype("float32")
                    
                    
                if day != 1: 
                    
                    # its same as above mapping but we are not subtracting day from date 
                    # same our dictionaries contain prev N day values for a given date 
                    
                    pos = (str(day) + "_" + (cur[str(i)] + "_" + \
                        (cur["date"]).astype(str)))
                        
                    neg = (str(day) + "_" + (cur[str(i) + str(i)] + "_" + \
                        (cur["date"]).astype(str)))
                        
                        
                    cur[f"{i}_POS_corr_MEAN_prev_{day}"] = pos.apply(lambda x: cor_mean_dict.get(x, np.nan)).astype("float32")
                    
                    cur[f"{i}_NEG_corr_MEAN_prev_{day}"] = neg.apply(lambda x: cor_mean_dict.get(x, np.nan)).astype("float32")
                        
                        
                    
                    cur[f"{i}_POS_corr_MEDIAN_prev_{day}"] = pos.apply(lambda x: cor_median_dict.get(x, np.nan)).astype("float32")
                    
                    cur[f"{i}_NEG_corr_MEDIAN_prev_{day}"] = neg.apply(lambda x: cor_median_dict.get(x, np.nan)).astype("float32")
                        
                    
                        
                    cur[f"{i}_POS_corr_STD_prev_{day}"] = pos.apply(lambda x: cor_std_dict.get(x, np.nan)).astype("float32")
                    
                    cur[f"{i}_NEG_corr_STD_prev_{day}"] = neg.apply(lambda x: cor_std_dict.get(x, np.nan)).astype("float32")
                        
                    
                    
                    cur[f"{i}_POS_corr_RANGE_prev_{day}"] = pos.apply(lambda x: cor_range_dict.get(x, np.nan)).astype("float32")
                    
                    cur[f"{i}_NEG_corr_RANGE_prev_{day}"] = neg.apply(lambda x: cor_range_dict.get(x, np.nan)).astype("float32")
                        
        
            print(i, "done")
            
        
        ###############################################################################
        ###############################################################################
        
        discount_cols = []
        for col in test:
            if re.search("disc", col):
                discount_cols.append(col)
        
        for col in discount_cols:
            cur.loc[cur[col]<0, col] = 0
        
        for col in discount_cols:
            if np.sum(cur[cur[col]<0][col]):
                print(col, " contains minus discount values")
            
        
        cur["Max_Discount"] = np.max(cur[discount_cols], axis = 1)
        
        ###############################################################################
    
        g_col = [(a,b,c) for a,b,c in zip(cur["unique_id"], cur["week"],(cur["Max_Discount"]>0).astype("float32"))]
    
        for col in ["sales", "availability"]:
            for agg in ["mean", "median", "std", "min"]:
                
                if col == "sales":
                    cur[col + "_" + "WEEKDAY_" + agg] = [
                        week_day_sales_dict[agg].get(
                            i, week_day_sales_dict[agg].get((i[0],i[1], not(i[2])), np.nan) )  for i in g_col]
                else:
                    cur[col + "_" + "WEEKDAY_" + agg] = [
                        week_day_avail_dict[agg].get(
                            i, week_day_avail_dict[agg].get((i[0],i[1], not(i[2])), np.nan) )  for i in g_col]
                
    
        
        ###############################################################################
        #train.sort_values(by = ["warehouse", "unique_id", "date"], inplace = True)
        #train.reset_index(drop = True, inplace = True)
        n = 1
        
        #train.sort_values(by = [ "product_unique_id", "date", "warehouse",], inplace = True)
        
        # prev day 
        
        
        cur[f"prev_{n}_sales"] = cur["sales"].shift(n)
        cur[f"prev_{n}_total_orders"] = cur["total_orders"].shift(n)
        cur[f"prev_{n}_sell_price_main"] = cur["sell_price_main"]
        cur[f"prev_{n}_Max_Discount"] = cur["Max_Discount"].shift(n)
        cur[f"prev_{n}_single_order_value"] = cur["single_order_value"].shift(n)
        
        
        # diff
        
        cur[f"prev_{n}_total_orders_diff"] = cur["total_orders"] - cur[f"prev_{n}_total_orders"]
        cur[f"prev_{n}_total_orders_diff_%"] = cur[f"prev_{n}_total_orders_diff"] / cur[f"prev_{n}_total_orders"].replace(0,1)
        
        cur[f"prev_{n}_sell_price_main_diff"] = cur["sell_price_main"] - cur[f"prev_{n}_sell_price_main"]
        cur[f"prev_{n}_sell_price_main_diff_%"] = cur[f"prev_{n}_sell_price_main_diff"] / cur[f"prev_{n}_sell_price_main"].replace(0,1)
        
        cur[f"prev_{n}_Max_Discount_diff"] = cur["Max_Discount"] - cur[f"prev_{n}_Max_Discount"]
        cur[f"prev_{n}_Max_Discount_diff_%"] = cur[f"prev_{n}_Max_Discount_diff"] / cur[f"prev_{n}_Max_Discount"].replace(0,1)
        
        cur[f"prev_{n}_single_order_value_diff"] = (cur["single_order_value"] - cur[f"prev_{n}_single_order_value"]).astype("float32")
        cur[f"prev_{n}_single_order_value_diff_%"] = cur[f"prev_{n}_single_order_value_diff"] / cur[f"prev_{n}_single_order_value"].replace(0,1)
    
        
        cur[f"prev_{n}_warehouse"] = cur["warehouse"].shift(n)
        cur[f"prev_{n}_unique_id"] = cur["unique_id"].shift(n)
        
        cur[f"prev_{n}_flag"] = (
            (cur[f"prev_{n}_warehouse"] == cur["warehouse"]) * 
            (cur[f"prev_{n}_unique_id"] == cur["unique_id"])
            ).astype(int)
        
        
        useless = [f"prev_{n}_warehouse", f"prev_{n}_unique_id", f"prev_{n}_flag"]
        
        
        set_null_cols = [
            f"prev_{n}_total_orders_diff", f"prev_{n}_total_orders_diff_%", 
            f"prev_{n}_sell_price_main_diff", f"prev_{n}_sell_price_main_diff_%",
            f"prev_{n}_Max_Discount_diff", f"prev_{n}_Max_Discount_diff_%",
            f"prev_{n}_sales", f"prev_{n}_total_orders", 
            f"prev_{n}_sell_price_main", f"prev_{n}_Max_Discount", 
            f"prev_{n}_single_order_value", f"prev_{n}_single_order_value_diff",
            f"prev_{n}_single_order_value_diff_%"
            ]
        
        cur.loc[(cur[f"prev_{n}_flag"]==0), set_null_cols] = np.nan
        
        for col in useless:
            del cur[col] 
        
        ###############################################################################
        
        N_shifts = [3,7,14,30]
        
        
        for n in range(2, max(N_shifts)+1):
            
            cur[f"prev_{n}_sales"] = cur["sales"].shift(n)
            cur[f"prev_{n}_total_orders"] = cur["total_orders"].shift(n)
            cur[f"prev_{n}_sell_price_main"] = cur["sell_price_main"].shift(n)
            cur[f"prev_{n}_Max_Discount"] = cur["Max_Discount"].shift(n)
            cur[f"prev_{n}_single_order_value"] = cur["single_order_value"].shift(n)
        
        
            # diff
        
            cur[f"prev_{n}_total_orders_diff"] = cur["total_orders"] - cur[f"prev_{n}_total_orders"]
            cur[f"prev_{n}_total_orders_diff_%"] = cur[f"prev_{n}_total_orders_diff"] / cur[f"prev_{n}_total_orders"].replace(0,1)
        
            cur[f"prev_{n}_sell_price_main_diff"] = cur["sell_price_main"] - cur[f"prev_{n}_sell_price_main"]
            cur[f"prev_{n}_sell_price_main_diff_%"] = cur[f"prev_{n}_sell_price_main_diff"] / cur[f"prev_{n}_sell_price_main"].replace(0,1)
        
            cur[f"prev_{n}_Max_Discount_diff"] = cur["Max_Discount"] - cur[f"prev_{n}_Max_Discount"]
            cur[f"prev_{n}_Max_Discount_diff_%"] = cur[f"prev_{n}_Max_Discount_diff"] / cur[f"prev_{n}_Max_Discount"].replace(0,1)
            
            cur[f"prev_{n}_single_order_value_diff"] = (cur["single_order_value"] - cur[f"prev_{n}_single_order_value"]).astype("float32")
            cur[f"prev_{n}_single_order_value_diff_%"] = cur[f"prev_{n}_single_order_value_diff"] / cur[f"prev_{n}_single_order_value"].replace(0,1)
    
    
            cur[f"prev_{n}_warehouse"] = cur["warehouse"].shift(n)
            cur[f"prev_{n}_unique_id"] = cur["unique_id"].shift(n)
        
        
        cols = [
            f"total_orders_diff", f"total_orders_diff_%", 
            f"sell_price_main_diff", f"sell_price_main_diff_%",
            f"Max_Discount_diff", f"Max_Discount_diff_%",
            f"sales", f"total_orders", 
            f"sell_price_main", f"Max_Discount", 
            f"single_order_value", f"single_order_value_diff",
            f"single_order_value_diff_%"
            ]
        
        
        ## 
        
        for col in cols:
            for c in N_shifts:
                cur_col_list = []
                
                sm_grt = 0
                sm_lt = 0
                sm_eq = 0
                
                c_col = f"prev_{c}_" + col
                for cn in range(1, c+1):
                    cn_col = f"prev_{cn}_" + col
                    cur_col_list.append(cn_col)
                    
                    if col != "sales":
                        sm_grt += cur[c_col] > cur[cn_col]
                        sm_lt += cur[c_col] < cur[cn_col]
                        sm_eq += cur[c_col] == cur[cn_col]
                    
                
                cur[f"prev_{c}_"+ col+ "_Mean" ] = np.mean(cur[cur_col_list], axis = 1).astype("float32")
                cur[f"prev_{c}_"+ col+ "_Median" ] = np.median(cur[cur_col_list], axis = 1).astype("float32")
                cur[f"prev_{c}_"+ col+ "_Std" ] = np.std(cur[cur_col_list], axis = 1).astype("float32")
                cur[f"prev_{c}_"+ col+ "_Min" ] = np.min(cur[cur_col_list], axis = 1).astype("float32")
                cur[f"prev_{c}_"+ col+ "_Max" ] = np.max(cur[cur_col_list], axis = 1).astype("float32")
    
                
                if (c>3) and (col in ["sales", "total_orders"]):
                    
                    cur[f"prev_{c}_"+ col+ "_kurtosis" ] = kurtosis(cur[cur_col_list].fillna(0), axis = 1).astype("float32")
                    cur[f"prev_{c}_"+ col+ "_skew" ] = skew(cur[cur_col_list].fillna(0), axis = 1).astype("float32")
                    cur[f"prev_{c}_"+ col+ "_IQR" ] = iqr(cur[cur_col_list].fillna(0), axis = 1).astype("float32")
                    cur[f"prev_{c}_"+ col+ "_65" ] = np.percentile(cur[cur_col_list], 65, axis = 1).astype("float32")
    
                
                if col != "sales":
                    cur[f"prev_{c}_"+ col+ "_Grt" ] = sm_grt
                    cur[f"prev_{c}_"+ col+ "_Grt_Norm" ] = (cur[f"prev_{c}_"+ col+ "_Grt" ] / c).astype("float32")
                    cur[f"prev_{c}_"+ col+ "_Lt" ] = sm_lt
                    cur[f"prev_{c}_"+ col+ "_Lt_Norm" ] = (cur[f"prev_{c}_"+ col+ "_Lt" ] / c).astype("float32")
                    cur[f"prev_{c}_"+ col+ "_Eq" ] = sm_eq
                    cur[f"prev_{c}_"+ col+ "_Eq_Norm" ] = (cur[f"prev_{c}_"+ col+ "_Eq" ] / c).astype("float32")
    
    
                
        
        ###############################################################################
        
        # Removing Unnecessary columns
        
        
        col_dict = {} 
        
        
        # Creating columns name list for each N, Used for setting values to NaN for this columns
        
        for n in N_shifts:
            nl = [
                f"prev_{n}_sales", f"prev_{n}_total_orders",
                f"prev_{n}_sell_price_main", f"prev_{n}_Max_Discount",
                f"prev_{n}_total_orders_diff", f"prev_{n}_total_orders_diff_%",
                f"prev_{n}_sell_price_main_diff", f"prev_{n}_sell_price_main_diff_%",
                f"prev_{n}_Max_Discount_diff", f"prev_{n}_Max_Discount_diff_%",
                f"prev_{n}_single_order_value", f"prev_{n}_single_order_value_diff",
                f"prev_{n}_single_order_value_diff_%"
            ]
            
            for col in cols:
                nl  += [
                    f"prev_{n}_"+ col+ "_Mean" ,
                    f"prev_{n}_"+ col+ "_Median",
                    f"prev_{n}_"+ col+ "_Std",
                    f"prev_{n}_"+ col+ "_Min",
                    f"prev_{n}_"+ col+ "_Max", 
            
                ] 
                
                if col != "sales":
                    
                    nl += [
                        f"prev_{n}_"+ col+ "_Grt", 
                        f"prev_{n}_"+ col+ "_Grt_Norm",
                        f"prev_{n}_"+ col+ "_Lt",
                        f"prev_{n}_"+ col+ "_Lt_Norm",
                        f"prev_{n}_"+ col+ "_Eq",
                        f"prev_{n}_"+ col+ "_Eq_Norm"
                    ]
    
                if (n>3) and (col in ["sales", "total_orders"]):
                
                    nl += [
                        f"prev_{n}_"+ col+ "_kurtosis", 
                        f"prev_{n}_"+ col+ "_skew", 
                        f"prev_{n}_"+ col+ "_IQR", 
                        f"prev_{n}_"+ col+ "_65" 
                        ]
                    
                col_dict[n] = nl
        
        
        ### Setting values to NaN where flag is False
        
        for n in N_shifts:
            cur[f"prev_{n}_flag"] = (
                (cur[f"prev_{n}_warehouse"] == cur["warehouse"]) * 
                (cur[f"prev_{n}_unique_id"] == cur["unique_id"])
                ).astype(int)
            
            if n==3:
                for nn in N_shifts:
                    cur.loc[(cur[f"prev_{n}_flag"]==0), col_dict[nn]] = np.nan
            
            if n==7:
                for nn in N_shifts[1:]:
                    cur.loc[(cur[f"prev_{n}_flag"]==0), col_dict[nn]] = np.nan
            
            if n==14:
                for nn in N_shifts[2:]:
                    cur.loc[(cur[f"prev_{n}_flag"]==0), col_dict[nn]] = np.nan
                    
            if n==30:
                for nn in N_shifts[3:]:
                    cur.loc[(cur[f"prev_{n}_flag"]==0), col_dict[nn]] = np.nan
        
        
        # New columns to be deleted which is used during N shift Aggregations like Mean of Last N days values
        
        to_drop1 = []
        for n in range(2, 31):
            if n in [3,7,14,30]:
                continue
            to_drop1 += [f"prev_{n}_sales", f"prev_{n}_total_orders", f"prev_{n}_sell_price_main", f"prev_{n}_Max_Discount",
            f"prev_{n}_total_orders_diff", f"prev_{n}_total_orders_diff_%", f"prev_{n}_sell_price_main_diff", 
            f"prev_{n}_sell_price_main_diff_%", f"prev_{n}_Max_Discount_diff", f"prev_{n}_Max_Discount_diff_%",
            f"prev_{n}_single_order_value", f"prev_{n}_single_order_value_diff",
            f"prev_{n}_single_order_value_diff_%",
            f"prev_{n}_warehouse", f"prev_{n}_unique_id"
            ]
        
         
        for col in to_drop1:
            del cur[col]
        
        #columns = train.columns
        
        new_drop = [
    
         'prev_3_flag',
         'prev_7_flag',
         'prev_14_flag',
         'prev_30_flag']
        
        drop = [
                "prev_3_warehouse",
                "prev_7_warehouse",
                "prev_14_warehouse",
                "prev_30_warehouse",
                "prev_3_unique_id",
                "prev_7_unique_id",
                "prev_14_unique_id",
                "prev_30_unique_id",
                ]
        
        for col in drop:
            del cur[col]
            
        for col in new_drop:
            del cur[col]
    
        
        
        import gc 
        
        del sm_grt, sm_lt, sm_eq
        gc.collect()
        
        
         
        ###############################################################################
        
        
        cur["disc_flag"] = (cur["Max_Discount"]==0).astype(int)
        #cur = cur.copy()
        
        g_col =  ( 
            cur["warehouse"].astype(str) + \
                "__"+ cur["unique_id"].astype(str) + \
            "__"+ cur["disc_flag"].astype(str)
            )
            
           
        g_col2 =  ( 
            cur["warehouse"].astype(str) + \
                "__"+ cur["unique_id"].astype(str)
            )
            
            
        #sales_dict = cur.groupby(g_col)["sales"].agg(["max","mean", "median", "std"]).to_dict()
        
        #sell_price_dict = cur.groupby(g_col2)["sell_price_main"].agg(["max", "mean", "median", "std"]).to_dict()
        
        
        for agg in ["max", "mean", "median", "std"]:
            for col in ["sales", "sell_price_main"]:
                
                if col == "sales":
                    cur[col + "_OVERALL_" + agg] = g_col.map(sales_dict[agg])
                    
                else: 
                    cur[col + "_OVERALL_" + agg] = g_col2.map(sell_price_dict[agg])
                    
        
        del cur["disc_flag"]
        
        ###############################################################################
        
        #dates = cur["date"].copy()
        #cur.drop(columns = to_drop, inplace = True)
        
        for col in ord_df:
            
            cur[col] = ord_df[col].astype("float32").copy()
        
        for col in L_df:
            cur[col] = L_df[col].astype("float32").copy()
            
        ###############################################################################
        
        grp = cur["unique_id"].astype(str) + "_" + cur["date"].astype(str)
        cur["Tot_Orders_Z"] = grp.map(tot_orders_z_dict)
        cur["Single_Order_Z"] = grp.map(single_order_z_dict)
        
        cur["holi_flag"] = cur["holiday_name"].notna().astype("float32")
        
        cur["holidayNM_next_1"] = cur["holi_flag"].shift(-1)
        cur["holidayNM_next_2"] = cur["holi_flag"].shift(-2)
        cur["holidayNM_prev_1"] = cur["holi_flag"].shift(1)
        cur["holidayNM_prev_2"] = cur["holi_flag"].shift(2)
        
        cur["holiday_next_1"] = cur["holiday"].shift(-1)
        cur["holiday_next_2"] = cur["holiday"].shift(-2)
        cur["holiday_prev_1"] = cur["holiday"].shift(1)
        cur["holiday_prev_2"] = cur["holiday"].shift(2)
        
        ###############################################################################
        target = "sales"
        
        ttest = cur[cur["sample"]=="test"].copy()
        
        del ttest[target]
        
        #______________________________________________________________________________#
        #______________________________________________________________________________#
    
    
        ################################################################################
        ###############################################################################
    
        ID = ttest["unique_id"].copy()
        
    
        #######################################################################################3
    
        splits = 5
        lpreds2 = np.zeros((splits, len(ttest)))
        lpath = f"/kaggle/input/rohlik-v{17}/v{17}/"
        
        for idx in range(splits):
            model = pickle.load(open(lpath + f"lgb{idx}.pkl", "rb"))
            features = [i for i in model.feature_name_]
            print(len(features))
            lpreds2[idx] = model.predict(ttest[features])#.clip(0)
    
    
        Bxpreds = np.zeros((splits, len(ttest)))
        xpath = f"/kaggle/input/rohlik-v{19}/v{19}/"
        
        for idx in range(splits):
            ttest["preds"] = lpreds2[idx]
            model = pickle.load(open(xpath + f"xgb{idx}.pkl", "rb"))
            features = [i for i in model.feature_names_in_]
            print(len(features))
            Bxpreds[idx] = model.predict(ttest[features], iteration_range = (0, model.best_iteration)).clip(0)
            del ttest["preds"]
    
    
    
        # BEST MLP + BXGB LB 18.21
        mpreds = np.zeros((splits, len(ttest)))
    
        features = pickle.load(open("/kaggle/input/mlpv2-misc/columns.pkl", "rb"))
    
    
        ttest1 = ttest.copy()
        
        for col in features:
            if col == "preds": continue
            ttest1[col] = ttest1[col].fillna(-100)
            
        n = 336
        hidden_units = [n*2, n*4, n*4, n*2]
        dropout_rates = [0.10, 0.19, 0.27, 0.23, 0.23]
        learning_rate = 1e-4
        
        model = create_mlp1(n, 1, hidden_units, dropout_rates, learning_rate)
    
        for col in features:
            if col == "preds": continue
            ttest1[col] = sc_dict[col].transform(ttest1[col].values.reshape(-1,1)).reshape(-1)
            
        path = "/kaggle/input/rohlik-v31/"

        ## THIS iS SIMPLE MLP, NOT LSTM, JUST A TYPO
        for idx in range(splits):
            ttest1["preds"] = Bxpreds[idx]
            ttest1["preds"] = sc_dict["preds"].transform(ttest1["preds"].values.reshape(-1,1)).reshape(-1)
            model.load_weights(path +f"LSTM_MLP_{idx}.weights.h5")
            mpreds[idx] = model.predict(ttest1[features].values).reshape(-1).clip(0)
            del ttest1["preds"]
    
        mpreds = np.mean(mpreds, axis = 0).clip(0)
     
    
        
    
    
    
        ###################################################################################
    
        splits = 5
        lpreds = np.zeros((splits, len(ttest)))
        lpath = f"/kaggle/input/rohlik-v{47}/"
        
        for idx in range(splits):
            model = pickle.load(open(lpath + f"LGB_H{idx}/LGB_H{idx}.pkl", "rb"))
            features = [i for i in model.feature_name_]
            print("LGBH:",len(features))
            lpreds[idx] = model.predict(ttest[features]).clip(0)
    
        xpreds = np.zeros((splits, len(ttest)))
        xpath = f"/kaggle/input/rohlik-v{48}/"
        
        for idx in range(splits):
            ttest["preds"] = lpreds[idx]
            model = pickle.load(open(xpath + f"XGB_H{idx}/XGB_H{idx}.pkl", "rb"))
            features = [i for i in model.feature_names_in_]
            print("XGBH:",len(features))
            xpreds[idx] = model.predict(ttest[features]).clip(0)
            del ttest["preds"]
    
    
    
    
        
        mpreds2 = np.zeros((splits, len(ttest)))
        mpreds3 = np.zeros((splits, len(ttest)))
    
        features = pickle.load(open("/kaggle/input/rohlik-v50/MLPV4_cols.pkl", "rb"))
    
    
        ttest1 = ttest.copy()
        
        for col in features:
            if col == "preds": continue
            ttest1[col] = ttest1[col].fillna(-100)
            
        n = 336
        hidden_units = [n*2, n*4, n*4, n*2]
        dropout_rates = [0.10, 0.19, 0.27, 0.23, 0.23]
        learning_rate = 1e-4
        
        model_g = create_mlp1_g(n, 1, hidden_units, dropout_rates, learning_rate)
        model = create_mlp1(n, 1, hidden_units, dropout_rates, learning_rate)
    
        for col in features:
            if col == "preds": continue
            ttest1[col] = sc_dictV4[col].transform(ttest1[col].values.reshape(-1,1)).reshape(-1)
    
    
        
        path = "/kaggle/input/rohlik-v55/"
        path2 = "/kaggle/input/rohlik-v50/"
        
        for idx in range(splits):
            ttest1["preds"] = xpreds[idx]
            ttest1["preds"] = sc_dictV4["preds"].transform(ttest1["preds"].values.reshape(-1,1)).reshape(-1)
            
            model_g.load_weights(path +f"MLPV6_{idx}.weights/MLPV6_{idx}.weights.h5")  # 18.22
            model.load_weights(path2 +f"MLPV4_{idx}.weights/MLPV4_{idx}.weights.h5")   # 18.23
            
            mpreds2[idx] = model_g.predict(ttest1[features].values).reshape(-1).clip(0)
            mpreds3[idx] = model.predict(ttest1[features].values).reshape(-1).clip(0)
            
            del ttest1["preds"]
            
    
        mpreds2 = np.mean(mpreds2, axis = 0).clip(0)
        mpreds3 = np.mean(mpreds3, axis = 0).clip(0)
    
    
    
    
        Bxpreds = np.mean(Bxpreds, axis = 0).clip(0)
        xpreds = np.mean(xpreds, axis = 0).clip(0)
        lpreds = np.mean(lpreds, axis = 0).clip(0)
        lpreds2 = np.mean(lpreds2, axis = 0).clip(0)
    
        preds = (
        (mpreds2 * 21) + (mpreds3 * 20) + (mpreds * 9) + 
        (Bxpreds * 18) +  (xpreds * 18) + 
        (lpreds * 6) + (lpreds2 * 8)
            ) / 100
    
    
    
        
        temp = pd.DataFrame()
        
        temp["unique_id"] = ID
        temp["id"] = temp["unique_id"].astype(str) + "_" + ttest["date"].astype(str)
        temp["date"] = ttest["date"]
        temp["sales"] = preds.copy()
        
        final_preds = pd.concat((final_preds, temp[["id", "sales"]]))

        if cur_day == 1:
            print(final_preds)
        
        concat_df = pd.merge(
            test[test["date_num"]==cur_day],
            temp[["unique_id", "date", "sales"]],
            how = "inner", 
            on = ["unique_id", "date"]
            )
        
        concat_df["sample"] = "train"
        
        train = pd.concat((train, concat_df), axis = 0)
        train.sort_values(by = ["warehouse", "unique_id", "date"], inplace = True)
        
        for k,v in zip((temp["unique_id"].astype(str) + "_" + temp["date"].astype(str)), temp["sales"]):
            cor_val_dict[k] = v
            
        print(f"Day {cur_day} Predicted!")

    return final_preds

In [2]:
sub1 = get_OLD_sub()

1 done
2 done
3 done
4 done
335
335
335
335
335
336
336
336
336
336
106/106 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step
106/106 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step
106/106 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step
106/106 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step
106/106 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step
LGBH: 335
LGBH: 335
LGBH: 335
LGBH: 335
LGBH: 335
XGBH: 336
XGBH: 336
XGBH: 336
XGBH: 336
XGBH: 336
106/106 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step
106/106 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step
106/106 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step
106/106 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step
106/106 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step
106/106 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step
106/106 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step
106/106 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step
106/106 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step
106/106 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step
                     id       sales
30        12_2024-06-03   33.846238
61        20_2024-06-03   35.219614
92        25_2024-06-03   71.076477
153       41_2024-06-03   64.029067
184       46_2024-06-0

In [3]:
import gc 
gc.collect()

17701

In [4]:
import numpy as np
import pandas as pd
import pickle 
import warnings
import os
import re
warnings.filterwarnings("ignore")

In [5]:
import tensorflow.keras.backend as K
import tensorflow.keras.layers as layers
import tensorflow as tf

In [6]:
def create_mlp1( num_columns, num_labels, hidden_units, dropout_rates):

    
        inp = tf.keras.layers.Input(shape = (num_columns,))
        #inp = tf.keras.layers.GaussianNoise(.03, seed = SEED)(inp)
        xinp = inp


        x = tf.keras.layers.BatchNormalization()(inp)
        x = layers.Dropout(dropout_rates[0])(x)
        for i in range(len(hidden_units)): 
            x = tf.keras.layers.Dense(hidden_units[i])(x)
            x = tf.keras.layers.BatchNormalization()(x)
            x = tf.keras.layers.Activation("swish")(x)
            x = tf.keras.layers.Dropout(dropout_rates[i+1])(x)  

            x = tf.keras.layers.Concatenate()([x, xinp])
            xinp = x


        x = tf.keras.layers.Dense(512)(x)

        x = tf.keras.layers.Activation("swish")(x)
    
        #x = tf.keras.layers.Dropout(.1)(x)
        x = tf.keras.layers.Dense(64)(x)
        x = tf.keras.layers.Activation("swish")(x)
    
        out = tf.keras.layers.Dense(num_labels)(x)
        
        
        model = tf.keras.models.Model(inputs = inp, outputs = [out])

    
        return model

In [7]:
MLP_feats = pickle.load(open("/kaggle/input/lgb2-all/MLP_710_NewParam/MLP_710_columns.pkl", "rb"))
sc_dict = pickle.load(open("/kaggle/input/lgb2-all/MLP_710_NewParam/MLP_710_Standard_Scaler.pkl", "rb"))

In [8]:
fruits = {'dinosaur eggs', 'star apple', 'nopal fruit', 'sunset apple', 'date plums', 'egg fruit', 'maracuya', 'lancetilla mango', 'red delicious apple', 'oeillade noire grape', 'marula', 'green raisins', 'umeboshi', 'limquat', 'quararibea', 'indian persimmon', 'cosmic crisp apple ', 'wild sugar apple', 'ambrosia melon', 'red mombin', 'valencia oranges', 'black sapote', 'pink pomelo', 'mangaba', 'damson plums', 'meyer lemon', 'rollinia', 'zill mango', 'alpine strawberry', 'limau bali', 'granny smith apple', 'pinova apples', 'batuan ', 'splendor apple', 'midyim berry', 'highbush cranberries', 'orient melon', 'zestar apple', 'xoconostle cactus fruit ', 'tachibana orange', 'african breadfruit', 'honeysuckles', 'fazli mango', 'northern spy apples', 'horned melon', 'red pear', 'zig zag vine fruit', 'jamun plum', 'green banana', 'fairchild tangerines', 'bilberries ', 'plums', 'annona', 'hala fruit', 'barbados cherries', 'zinfandel grape', 'durian', 'passion fruit', 'oriental cherry', 'ortanique', 'marang', 'kakadu plum', 'cabernet sauvignon grape ', 'rough lemon', 'wax gourd', 'wampee', 'winter squash', 'keitt mango', 'o’henry peach', 'boysenberry', 'pink lady apples', 'persian limes', 'ogallala strawberry', 'orange', 'osteen mango', 'desert fig', 'gooseberry', 'hog plums', 'ximenia caffra ', 'ichigo', 'acerola cherries', 'wild water lemon', 'blue watermelon', 'kutjera fruit', 'rangpur lime', 'rambai', 'pequi fruit', 'caper berry', 'honeyberries', 'rose apple', 'juicy tomatoes', 'brazilian guava ', 'nectacotum', 'parsonage pears', 'himalayan mulberries', 'eggplant', 'papaya', 'quandong', 'korean pear', 'lady apple', 'zarzamora', 'wild strawberry', 'cedar bay cherry ', 'zalzalak', 'ogeechee lime', 'youngberry', 'bottle gourd', 'cranberry', 'blood orange', 'fibrous satinash fruit', 'okuzgozu grape', 'pacific rose apple', 'batuan', 'courgette', 'west indian cherry', 'jazz apples', 'sudachi', 'lippens mango', 'langsat', 'macoun apple', 'grapefruit', 'avocado', 'nungu fruit', 'jackfruit', 'huckleberry', 'usuma fruit', 'queen’s forelle pear', 'kasturi mango', 'orient pear', 'pumpkins', 'jostaberry', 'vaccarese grapes', 'petit rouge grapes', 'hyuganatsu', 'otaheite gooseberry', 'akebi fruit', 'topaz apple', 'oso grande strawberry', 'mayan nut', 'canistel fruit', 'zweigelt grape', 'juniper berries', 'ensete', 'orin apple', 'tourist pineapple', 'elephant apple (chalta)', 'dragon fruit', 'indian prune', 'le conte pear', 'ground plum', 'coco plum ', 'green lemon', 'evergreen huckleberry', 'miracle fruit', 'satsuma mandarin', 'coconut', 'hackberry', 'fascell mango', 'ziziphus mauritiana', 'ya pear', 'mcintosh apple', 'pink pineapple', 'yellow guava ', 'husk tomatoes', 'nepali hog plums', 'may pride peaches', 'red bayberry', 'red passion fruit', 'black currant', 'urava fruit ', 'yayat palm', 'blue java banana ', 'limeberry', 'asam kumbang', 'peach palm fruit', 'lemon aspen', 'sugar plum', 'galia melon', 'xigua fruit ', 'marsh pink grapefruit', 'ambrosia apples', 'olive', 'yellow dragon fruit', 'nam dok mai', 'nance', 'opuntia', 'desert quandong', 'japanese persimmon', 'amla', 'rosigold mango', 'java apple', 'louise bonne of jersey pear', 'tommy atkins mango', 'ita palm fruit', 'lima orange', 'lemato', 'madison peach', 'victoria plum', 'farkleberry', 'newton pippin apple', 'pomegranate', 'forest strawberries', 'red banana', 'green bell pepper', 'indian jujube fruit', 'mango', 'annatto', 'queen apple', 'pineapple', 'cortland apple ', 'sugar berry', 'darwin’s barberries', 'betel nut', 'white currant', 'five flavor berry', 'langra mango', 'camu camu berry', 'pinot noir grapes', 'ozark gold apple', 'cleopatra mandarin', 'lime', 'mexican limes ', 'hawthorn fruit', 'etrog', 'olallieberry', 'governor’s plum', 'quince ', 'lilly pilly', 'xarel-lo ', 'umbu fruit', 'york apple', 'indian almond fruit', 'yellow strawberry', 'emblica', 'pie pumpkins', 'plumcots', 'honey crisp apples', 'dabai fruit', 'entawak', 'oval kumquat', 'blue dragon fruit', 'raspberry', 'imbu ', 'nectarines', 'imbe', 'wild peach', 'green fig', 'rockmelon', 'kvede', 'johannisbeere', 'ozark beauty strawberry', 'false mastic fruit', 'zabergau reinette apple', 'yellow chili pepper', 'velvet tamarind', 'cherry', 'pili fruit', 'yellow plum', 'riberry', 'indonesian lime', 'mandarin', 'green adjou pear', 'taylor’s gold pear', 'uvilla fruit', 'marionberry', 'yayat', 'cornelian cherry', 'may apple', 'rose hip', 'voavanga fruit', 'fox grapes', 'jelly palm fruit', 'oroblanco', 'kaffir lime', 'wahoo', 'ximenia americana ', 'calabash', 'iboga', 'white mulberries', 'nuts', 'zawngtah', 'strawberry guava ', 'red mulberry', 'ugli fruit', 'fuji apples', 'williams pear', 'highbush blueberries', 'yellow sapote', 'black cherries', 'yucca', 'zwetschge', 'texas persimmon', 'kei apple.', 'green tomato', 'tomatillo', 'raspuri mango', 'blueberries', 'nagami kumquat', 'breadfruit ', 'tart cherry', 'lapsi', 'walnut', 'discovery apple', 'rumdul', 'maqui', 'van dyke mango', 'nashi pears', 'churchland pear', 'peaches', 'canary fruit', 'jatoba t', 'junglesop fruit', 'icacina', 'pineberries', 'monster deliciosa', 'brazil nut', 'cantaloupe', 'green grapes', 'pluots', 'citron', 'emu apple fruit', 'beechnut', 'european pear', 'yumberry', 'oullins gage plum', 'carambola', 'enterprise apple', 'madrono', 'figs', 'amanatsu oranges', 'chinese quince ', 'shonan gold ', 'merlot grape', 'naranjilla', 'long neck avocado', 'pigeon plum', 'orangelo', 'kanzi apple', 'green egg squash ', 'mammee apple', 'yemenite citron', 'strawberry', 'minneolo tangelo', 'early girl tomato', 'rajka apples', 'green gage plum', 'valencia pride mango', 'villafranca lemon', 'water apple', 'redcurrant', 'blood lime', 'mexican lime', 'volkamer lemon ', 'vanilla', 'early gold mango', 'white-flowered gourd', 'kiwano', 'apricots', 'terap', 'wild banana', 'double coconut', 'bacuri fruit', 'green strawberry', 'loquat', 'beach plums', 'nageia', 'finger lime', 'blue pearmain apple', 'ilama', 'chinese white pear ', 'ugni berry', 'western hackberry', 'tropical apricot', 'asian pears', 'xinomavro ', 'lemon', 'trifoliate orange', 'black raspberries', 'kyoho grape', 'gamboge', 'yangmei', 'pomato', 'dangle berry', 'indian sherbet berry', 'lady finger banana', 'sour cherry', 'ice cream bean fruit', 'lemon drop melon', 'kaywa', 'red huckleberry', 'umbra fruit', 'lambkin melon', 'kwai muk', 'paw paw', 'fukushu kumquat', 'opal plum', 'wax apple', 'quinault strawberry', 'honey locust', 'persimmons', 'red grape', 'kawakawa', 'golden apple', 'african mango', 'yali pear', 'prickly pears', 'sweet lime', 'carob ', 'yellow watermelon', 'bell peppers ', 'giant lau lau', 'genip', 'marisol clementine', 'jujube fruit', 'chinoto sour orange ', 'karonda berry', 'dracontomelon', 'melon pear fruit', 'salak ', 'kabosu', 'heirloom tomatoes', 'querina apples', 'arava melon', 'clementine', 'yuzu', 'feijoa', 'xerophyte', 'guavaberry', 'biriba', 'wild bacher grape', 'wild lowbush blueberry', 'bearberries', 'orlando tangelo', 'nutmeg', 'crab apple', 'fe’i banana', 'nere fruit', 'sweet granadilla', 'pink pearl apples', 'dekopon', 'tamarillo', 'mamey sapote', 'pitaya', 'wealthy apple', 'gorham pear', 'young mango', 'darling plum', 'zhe', 'margil apples', 'duku fruit', 'blue olive', 'cavendish banana ', 'indian gooseberry', 'pigface fruit', 'crimson gold apple', 'key lime', 'mock strawberry', 'neem', 'golden delicious apple', 'madrono ', 'grape', 'black mulberry ', 'pears', 'queen anne cherry', 'dates', 'oregon grape', 'knobby russet apple', 'watermelon', 'wild orange', 'hazelnuts', 'ice apple', 'cempedak ', 'star fruit', 'huito', 'tangerine', 'wild mangosteen', 'hawaiian mountain apples', 'tompkins king apple', 'mamoncillo', 'flatwoods plum', 'ambarella', 'wolf berry', 'loganberry', 'granadilla', 'peumo fruit', 'balsam apples ', 'yellow apple', 'naval oranges', 'plantains', 'grosella negra', 'queen tahiti pineapple', 'opal apple', 'crimson delight apple', 'guarana.', 'florida cherry', 'honeydew melon', 'bitter melon', 'rhobs el arsa', 'japanese plum', 'golden kiwi', 'blackberries ', 'mangosteen', 'conference pear', 'narenj', 'hardy kiwi', 'vespolina grape', 'lucuma', 'sloe', 'yellow bell peppers', 'kiwi ', 'zebra melon', 'tangelo', 'lodi apple', 'greek fig', 'gala apple', 'maypop', 'naartjie', 'blue sausage fruit', 'kumquats ', 'lord lambourne apple', 'guava', 'desert lime', 'pomelo', 'tayberry', 'custard apple', 'sand cherry', 'okra', 'cucumber', 'grumichama', 'dewberries', 'illawarra plum', 'natal plum', 'black apples', 'rambutan', 'xylocarp ', 'goji berry', 'wild lemon', 'roselle', 'elands sour fig', 'vicar of winkfield pear', 'lingonberry', 'raisins', 'nectacot', 'lablab fruit', 'sugar baby watermelon ', 'velvet apple', 'ponderosa lemons', 'oil palm', 'prunes', 'rocha pear', 'rata', 'dodder laurel', 'soncoya', 'emu berry fruit', 'xanthium strumarium ', 'sour plum ', 'crispin apple', 'rumbus parviflorus', 'kaki persimmon', 'ramontchi', 'barberries ', 'goumi', 'babaco', 'yellow passion fruit ', 'dessert banana', 'macadamia nut', 'brush cherry', 'green apple', 'blue marble fruit', 'pecan', 'elderberry', 'incaberry', 'zante currant', 'santol', 'ububese ', 'davidson’s plums', 'thimberry', 'korlan', 'water chestnut', 'dead man’s finger', 'winter melon', 'grand nain banana', 'nocera grape', 'serviceberry ', 'white aspen', 'amaou strawberries', 'soursop', 'liberty apple', 'zierfandler grape', 'kahikatea', 'eastern hawthorn fruit', 'palestinian sweet limes', 'yunnan hackberry', 'saigon mango.', 'kowai', 'icaco'}

In [9]:
vegetables = {'shallot', 'bean sprout', 'cheddar cauliflower', 'molokhia', 'udupi mattu gulla eggplant.', 'collard greens', 'leek', 'cherry pepper', 'eddoe', 'fava bean', 'zephyr squash ', 'purple carrot', 'mushroom ', 'lima bean ', 'butternut squash', 'ulluco', 'xanthosoma brasiliense', 'fluted pumpkin', 'ash gourd', 'flint corn', 'chamomile', 'vidalia onion', 'yam', 'all blue potato', 'new zealand spinach', 'broccoflower ', 'radish', 'gigante bean', 'nettle', 'russian banana potato', 'stokes purple sweet potato', 'garlic', 'celery  ', 'arugula', 'lavender', 'black radish', 'taro', 'hot pepper', 'bitter melon', 'chickpeas', 'soybean', 'hubbard squash', 'radicchio', 'navy bean', 'eggplant ', 'flat beans', 'topinambur', 'gold rush squash', 'snap pea', 'xanthosoma sagittifolium', 'great northern bean', 'savoy cabbage', 'kohlrabi', 'mallow', 'onion', 'sweet corn', 'chives', 'yellow onion', 'white bell pepper ', 'daikon', 'red onion', 'carnival squash', 'purple lettuce', 'boniato', 'tat soi', 'french sorrel', 'tabasco pepper', 'red carrot', 'red kuri squash', 'velvet bean', 'corn', 'dill', 'yellow summer squash', 'anise', 'red wing onion', 'spinach', 'belgian endive', 'corms', 'yukon gold potatoes', 'salsify ', 'dickinson pumpkin', 'white asparagus', 'red cabbage', 'arame', 'upland cress', 'cannellini bean', 'chili pepper', 'purple kale', 'bamboo shoot', 'pinto bean', 'red gold potato', 'white 7 pot pepper', 'azuki bean', 'yukon gold potato', 'french bean', 'honeynut squash', 'coriander', 'hubbard squash.', 'purple tomato', 'okra', 'spaghetti squash', 'lentil', 'cucumber', 'xà lách', 'delicata', 'new potato', 'jerusalem artichoke', 'flat italian onion', 'red bell pepper ', 'torpedo onion', 'peruvian purple potato', 'flat white boer pumpkin', 'artichoke', 'patty pan squash.', 'pea', 'jicama', 'asian greens', 'mung bean', 'orange tomato', 'split pea', 'sweet potato', 'cauliflower', 'german butterball potato', 'scallion ', 'futsu squash', 'kale', 'field cucumber', 'parsnip', 'fat hen', 'pepper', 'baby boo pumpkin', 'bell pepper', 'fiddlehead', 'chard ', 'kidney bean', 'malanga', 'squash', 'skirret', 'carrot', 'purple majesty potato', 'cabbage', 'red bliss potato', 'amaranth', 'vivaldi potato', 'mangelwurzel', 'purple bell pepper ', 'black bean', 'ambercup squash', 'cayenne pepper', 'oregano', 'gem squash', 'lettuce', 'broad-leaf arrowhead', 'arrowroot', 'mustard greens', 'adzuki bean', 'white corn', 'basil', 'wax bean', 'frisée', 'lumina pumpkin', 'arikara squash ', 'marjoram', 'peanut', 'purple radish', 'red warty thing squash', 'purple brussels sprout', 'broad beans', 'fiorentino tomato', 'yellow tomato', 'banana squash', 'red russian kale', 'bintje potato', 'alfalfa sprouts', 'orange bell pepper', 'purple corn', 'celeriac', 'purple cabbage', 'rhubarb', 'avocado', 'black-eyed pea', 'paprika', 'vitelotte potato', 'konjac', 'japanese sweet potato', 'ginger', 'oak leaf lettuce ', 'wasabi', 'morels', 'broccoli ', 'pie pumpkin', 'borage', 'rosemary', 'borlotti bean', 'xi yang ca', 'runner bean', 'green bean', 'water chestnut', 'zucchini ', 'ube', 'mangetout', 'tomato', 'brussels sprout', 'broccolini ', 'cymbopogon', 'purple cauliflower', 'turnip', 'endive', 'watercress', 'rutabaga', 'purple asparagus', 'acorn squash', 'caraway', 'tomatillo', 'ahipa', 'bok choy', 'kelp', 'black salsify', 'beet', 'asparagus', 'urad bean', 'honey bear squash ', 'aonori', 'habanero pepper', 'xi lan hua', 'pattypan squash', 'butterhead lettuce', 'potato', 'fennel', 'boston morrow squash', 'miner’s lettuce', 'jalapeño', 'fayot bean', 'purple string bean', 'burdock root', 'butter bean', 'thyme', 'banana pepper'}

In [10]:
path = "/kaggle/input/rohlik-sales-forecasting-challenge-v2/"
#model_str = "LGB2_PREV120_1K_"
#features = pickle.load(open("/kaggle/input/fast-miscs/LGB2_MORERANK_0/LGB2_MORERANK_0.pkl", "rb")).feature_name_
features = pickle.load(open("/kaggle/input/lgb2-all/LGB2_709_NewParam/LGB2_709_NewParam_0.pkl", "rb")).feature_name_

warehouse_map = {
    'Budapest_1': 0, 'Prague_2': 2,
    'Brno_1': 4, 'Prague_1': 1, 'Prague_3': 3,
     'Munich_1': 5, 'Frankfurt_1': 6
     }

train = pd.read_csv(path + "sales_train.csv")
test = pd.read_csv(path + "sales_test.csv")
inventory = pd.read_csv(path + "inventory.csv")
calendar = pd.read_csv(path + "calendar.csv")

train = pd.read_csv(path + "sales_train.csv")

cor_id_dict_pos = pickle.load( open("/kaggle/input/fast-miscs/cor_id_dict_posV2.pkl", "rb"))
cor_id_dict_neg = pickle.load( open("/kaggle/input/fast-miscs/cor_id_dict_negV2.pkl", "rb"))


###############################################################################

# for LGB2 Noise

train["warehouse"] = train.warehouse.map(warehouse_map)
train.sort_values(by = ["warehouse", "unique_id", "date"], inplace = True)
train.dropna(subset = "sales", inplace = True)
train.reset_index(drop = True, inplace = True)


idx_to_wh_map = {v:k for k,v in warehouse_map.items()}

train["warehouse"] = train["warehouse"].map(idx_to_wh_map)


train["sales"] = pickle.load(open("/kaggle/input/fast-miscs/sale_noise.pkl", "rb")).reset_index(drop = True).astype("float32")

cor_val_dict = pickle.load(open("/kaggle/input/fast-miscs/MISCS/cor_val_dict.pkl", "rb"))
cor_mean_dict = pickle.load(open("/kaggle/input/fast-miscs/MISCS/INF_cor_mean_dict.pkl", "rb"))
cor_median_dict = pickle.load(open("/kaggle/input/fast-miscs/MISCS/IN_cor_median_dict.pkl", "rb"))
cor_std_dict = pickle.load(open("/kaggle/input/fast-miscs/MISCS/IN_cor_std_dict.pkl", "rb"))
cor_range_dict = pickle.load(open("/kaggle/input/fast-miscs/MISCS/INF_cor_range_dict.pkl", "rb"))


sales_dict = pickle.load(open("/kaggle/input/lgb2-all/sales_dict_noise_SCIPY.pkl", "rb"))
sell_price_dict = pickle.load(open("/kaggle/input/fast-miscs/MISCS/sell_price_dict.pkl", "rb"))

discount_dict = pickle.load(open("/kaggle/input/fast-miscs/MISCS/discount_dict.pkl", "rb"))

ord_enc = pickle.load(open("/kaggle/input/fast-miscs/MISCS/ord_enc.pkl", "rb"))


#tot_order_RANK_perc = pickle.load(open("/kaggle/input/fast-miscs/MISCS/tot_order_RANK_perc.pkl", "rb"))
#sales_RANK_perc = pickle.load(open("/kaggle/input/fast-miscs/MISCS/sales_RANK_perc.pkl", "rb")) 

#sales_RANK_DF = train[["unique_id", "date", "sales"]]


In [11]:
final_preds = pd.DataFrame(columns = ["id", "sales"])


    
# DATE
train["date"] = pd.DatetimeIndex(train["date"])

train["day"] = train["date"].dt.day
train["week"] = train["date"].dt.dayofweek
train["week_of_year"] = train["date"].dt.isocalendar().week
train["month"] = train["date"].dt.month
train["quarter"] = train["date"].dt.quarter
train["year"] = train["date"].dt.year - 2020
train["day_of_year"] = train["date"].dt.dayofyear
train["day_num_from_start"] = (train["year"]  * 366) + train["day_of_year"]
train["week_num_from_start"] = (train["year"]  * 52) + train["week_of_year"]
train["month_num_from_start"] = (train["year"]  * 12) + train["month"]
train["quarter_num_from_start"] = (train["year"]  * 4) + train["quarter"]
train["week_of_month"] = train["day"] // 7

###############################################################################

## when train is sorted
rnk = train.groupby(["warehouse", "unique_id"])["day_num_from_start"].transform("rank", ascending = False)

## need last 42 days records, since I am using last 42 days aggregations
train = train.loc[rnk<=42]
train["rank"] = train.groupby(["warehouse", "unique_id"])["day_num_from_start"].transform("rank", ascending = False)

###############################################################################

# DATE
test["date"] = pd.DatetimeIndex(test["date"])

test["day"] = test["date"].dt.day
test["week"] = test["date"].dt.dayofweek
test["week_of_year"] = test["date"].dt.isocalendar().week
test["month"] = test["date"].dt.month
test["quarter"] = test["date"].dt.quarter
test["year"] = test["date"].dt.year - 2020
test["day_of_year"] = test["date"].dt.dayofyear
test["day_num_from_start"] = (test["year"]  * 366) + test["day_of_year"]
test["week_num_from_start"] = (test["year"]  * 52) + test["week_of_year"]
test["month_num_from_start"] = (test["year"]  * 12) + test["month"]
test["quarter_num_from_start"] = (test["year"]  * 4) + test["quarter"]
test["week_of_month"] = test["day"] // 7

###############################################################################

train["sample"] = "train"
test["sample"] = "test"

###############################################################################

test["date_num"] = test["date"].transform("rank", method = "dense")

In [12]:
for cur_day in range(1, 15):
    
    ###############################################################################

    cur = test[test["date_num"]==cur_day].copy()
    
    cur = pd.concat((train, cur), axis = 0)
    
    cur.sort_values(by = ["warehouse", "unique_id", "date"], inplace = True)
    cur.reset_index(drop = True, inplace = True)
    
    cur_drop = ["sample", "date_num"]
    
    
    ###############################################################################
    
    cur = pd.merge(cur, inventory, on = "unique_id", how = "left")
    cur.drop(columns = "warehouse_y", inplace = True)
    cur.rename(columns = {"warehouse_x": "warehouse"}, inplace = True)
    
    #cur["date"] = pd.DatetimeIndex(cur["date"])
    calendar["date"] = pd.DatetimeIndex(calendar["date"])
    
    # idx_to_wh_map = {v:k for k,v in warehouse_map.items()}
    
    # cur["warehouse"] = cur["warehouse"].map(idx_to_wh_map)
    
    cur = pd.merge(cur, calendar, on = ["date", "warehouse"] , how = "left")
    
    cur["warehouse"] = cur["warehouse"].map(warehouse_map)
    
    
    names = cur["name"].str.split("_", expand = True)
    
    cur["prod_name"] = names[0]
    
    def return_fruit(x):
        if x.lower() in fruits:
            return x 
        return np.nan
    
    def return_veggy(x):
        if x.lower() in vegetables:
            return x 
        return np.nan
    
    
    cur["fruit"] = names[0].apply(return_fruit)
    
    
    cur["veggy"] = names[0].apply(return_veggy)
    
    ##############################################################################

    # this is not exactly single order value, as  
    # it is total orders placed for all products for that day for that warehouse 

    cur["single_order_value"] = cur["total_orders"] / cur["sell_price_main"]

    cur["single_order_value"] = cur["single_order_value"].astype("float32")

    ###############################################################################
    
    ###############################################################################
    
    to_drop = ["holiday_name", "date", 
               "name", "L1_category_name_en", "L2_category_name_en", "L3_category_name_en",
               "L4_category_name_en",  "fruit", "veggy", "availability"]
    
    ###############################################################################
    
    warehouse_map = {
        'Budapest_1': 0, 'Prague_2': 2,
        'Brno_1': 4, 'Prague_1': 1, 'Prague_3': 3,
         'Munich_1': 5, 'Frankfurt_1': 6
         }
    
    #cur["warehouse"] = cur["warehouse"].map(warehouse_map)
    
    ###############################################################################
    
    # L 
    
    Ls = ["L2_category_name_en", "L3_category_name_en", "L4_category_name_en"]
    L_df = {}
    L_sum = 0
    
    for col in Ls:
         L_df[col + "grp3"] = cur[col].str.split("_", expand = True)[2].astype("float32")
         L_sum += L_df[col + "grp3"].astype("float32")
    
    L_df["L_sum"] = L_sum 
    del L_sum
    
    ###############################################################################
    
    
    # Ordinal Encoder
    
    ord_cols = ["name", "fruit", "veggy", "L2_category_name_en", "L3_category_name_en", "L4_category_name_en"]
    
    ord_df = ord_enc.transform(cur[ord_cols])
    
    ord_df = pd.DataFrame(ord_df, columns = [i + "_enc" for i in ord_cols])
    
    ###############################################################################
    
    for col in ord_df:
        
        cur[col] = ord_df[col].astype("float32").copy()
    
    for col in L_df:
        cur[col] = L_df[col].astype("float32").copy()
        
    del ord_df, L_df
    
    ###############################################################################
    ###############################################################################
    
    discount_cols = []
    for col in test:
        if re.search("disc", col):
            discount_cols.append(col)
    
    for col in discount_cols:
        cur.loc[cur[col]<0, col] = 0
    
    for col in discount_cols:
        if np.sum(cur[cur[col]<0][col]):
            print(col, " contains minus discount values")
        
    
    cur["Max_Discount"] = np.max(cur[discount_cols], axis = 1)
    
    
    ###########################################################################
    ###############################################################################
    
    ###########################################################################
    ###########################################################################
    
    
    for day in [3, 5, 7, 14, 21, 28,30, 35]:
        

        mn = cur.groupby(["warehouse","unique_id"])["sales"].shift(1).rolling(window = day).agg("mean").astype("float32")
        
        for k,v in zip((cur["unique_id"].astype(str) + "_" + cur["date"].astype(str)), mn):
            cor_mean_dict[str(day) + "_" + k] = v
            

        med = cur.groupby(["warehouse","unique_id"])["sales"].shift(1).rolling(window = day).agg("median").astype("float32")
        
        for k,v in zip((cur["unique_id"].astype(str) + "_" + cur["date"].astype(str)), med):
            cor_median_dict[str(day) + "_" + k] = v
            

        std = cur.groupby(["warehouse","unique_id"])["sales"].shift(1).rolling(window = day).agg("std").astype("float32")
        
        for k,v in zip((cur["unique_id"].astype(str) + "_" + cur["date"].astype(str)), std):
            cor_std_dict[str(day) + "_" + k] = v
            

        Max = cur.groupby(["warehouse","unique_id"])["sales"].shift(1).rolling(window = day).agg("max").astype("float32")
        Min = cur.groupby(["warehouse","unique_id"])["sales"].shift(1).rolling(window = day).agg("min").astype("float32")
        ran = (Max - Min).astype("float32")
        
        for k,v in zip((cur["unique_id"].astype(str) + "_" + cur["date"].astype(str)), ran):
            cor_range_dict[str(day) + "_" + k] = v
            

    del Max, Min
    del ran, std, med, mn

    
    cur["1"] = cur["unique_id"].apply(lambda x: cor_id_dict_pos[x][0] if cor_id_dict_pos[x] else np.nan).apply(lambda x: f"{x:.0f}")
    cur["2"] = cur["unique_id"].apply(lambda x: cor_id_dict_pos[x][1] if cor_id_dict_pos[x] else np.nan).apply(lambda x: f"{x:.0f}")
    cur["3"] = cur["unique_id"].apply(lambda x: cor_id_dict_pos[x][2] if cor_id_dict_pos[x] else np.nan).apply(lambda x: f"{x:.0f}")
    cur["4"] = cur["unique_id"].apply(lambda x: cor_id_dict_pos[x][3] if cor_id_dict_pos[x] else np.nan).apply(lambda x: f"{x:.0f}")
    cur["5"] = cur["unique_id"].apply(lambda x: cor_id_dict_pos[x][4] if cor_id_dict_pos[x] else np.nan).apply(lambda x: f"{x:.0f}")
    cur["6"] = cur["unique_id"].apply(lambda x: cor_id_dict_pos[x][5] if cor_id_dict_pos[x] else np.nan).apply(lambda x: f"{x:.0f}")
    cur["7"] = cur["unique_id"].apply(lambda x: cor_id_dict_pos[x][6] if cor_id_dict_pos[x] else np.nan).apply(lambda x: f"{x:.0f}")
    cur["8"] = cur["unique_id"].apply(lambda x: cor_id_dict_pos[x][7] if cor_id_dict_pos[x] else np.nan).apply(lambda x: f"{x:.0f}")
    cur["9"] = cur["unique_id"].apply(lambda x: cor_id_dict_pos[x][8] if cor_id_dict_pos[x] else np.nan).apply(lambda x: f"{x:.0f}")


    cur["11"] = cur["unique_id"].apply(lambda x: cor_id_dict_neg[x][0] if cor_id_dict_neg[x] else np.nan).apply(lambda x: f"{x:.0f}")
    cur["22"] = cur["unique_id"].apply(lambda x: cor_id_dict_neg[x][1] if cor_id_dict_neg[x] else np.nan).apply(lambda x: f"{x:.0f}")
    cur["33"] = cur["unique_id"].apply(lambda x: cor_id_dict_neg[x][2] if cor_id_dict_neg[x] else np.nan).apply(lambda x: f"{x:.0f}")
    cur["44"] = cur["unique_id"].apply(lambda x: cor_id_dict_neg[x][3] if cor_id_dict_neg[x] else np.nan).apply(lambda x: f"{x:.0f}")
    
    
    
    
    for i in range(1, 10):
        
        for day in [1,3,5,7,14,21,28,30,35]:
            
            
            
            cur[f"{i}_POS_corr_prev_{day}"] = ((cur[str(i)] + "_" + \
                (cur["date"] - pd.DateOffset(day)).astype(str)).apply(lambda x: cor_val_dict.get(x, np.nan))).astype("float32")
            
            if i < 5:
                cur[f"{i}_NEG_corr_prev_{day}"] = ((cur[str(i) + str(i)] + "_" + \
                    (cur["date"] - pd.DateOffset(day)).astype(str)).apply(lambda x: cor_val_dict.get(x, np.nan))).astype("float32")
                    
                
            if day != 1: 
                
                # its same as above mapping but we are not subtracting day from date 
                # same our dictionaries contain prev N day values for a given date 
                
                pos = (str(day) + "_" + (cur[str(i)] + "_" + \
                    (cur["date"]).astype(str)))
                
                if i < 5:
                    neg = (str(day) + "_" + (cur[str(i) + str(i)] + "_" + \
                        (cur["date"]).astype(str)))
                    
                    
                cur[f"{i}_POS_corr_MEAN_prev_{day}"] = pos.apply(lambda x: cor_mean_dict.get(x, np.nan)).astype("float32")
                cur[f"{i}_POS_corr_MEDIAN_prev_{day}"] = pos.apply(lambda x: cor_median_dict.get(x, np.nan)).astype("float32")
                cur[f"{i}_POS_corr_STD_prev_{day}"] = pos.apply(lambda x: cor_std_dict.get(x, np.nan)).astype("float32")
                cur[f"{i}_POS_corr_RANGE_prev_{day}"] = pos.apply(lambda x: cor_range_dict.get(x, np.nan)).astype("float32")
                
                
                if i < 5:
                    cur[f"{i}_NEG_corr_MEAN_prev_{day}"] = neg.apply(lambda x: cor_mean_dict.get(x, np.nan)).astype("float32")
                    cur[f"{i}_NEG_corr_MEDIAN_prev_{day}"] = neg.apply(lambda x: cor_median_dict.get(x, np.nan)).astype("float32")
                    cur[f"{i}_NEG_corr_STD_prev_{day}"] = neg.apply(lambda x: cor_std_dict.get(x, np.nan)).astype("float32")
                    cur[f"{i}_NEG_corr_RANGE_prev_{day}"] = neg.apply(lambda x: cor_range_dict.get(x, np.nan)).astype("float32")

        print(i, "done")
    
    
    ###########################################################################
    ###########################################################################
    
    #cur.sort_values(by = ["warehouse", "unique_id", "date"], inplace = True)
    #cur.reset_index(drop = True, inplace = True)
    n = 1
    
    #cur.sort_values(by = [ "product_unique_id", "date", "warehouse",], inplace = True)
    
    # prev day 
    
    
    cur[f"prev_{n}_sales"] = cur["sales"].shift(n)
    cur[f"prev_{n}_total_orders"] = cur["total_orders"].shift(n)
    cur[f"prev_{n}_sell_price_main"] = cur["sell_price_main"]
    cur[f"prev_{n}_Max_Discount"] = cur["Max_Discount"].shift(n)
    cur[f"prev_{n}_single_order_value"] = cur["single_order_value"].shift(n)
    
    
    # diff
    
    cur[f"prev_{n}_total_orders_diff"] = cur["total_orders"] - cur[f"prev_{n}_total_orders"]
    cur[f"prev_{n}_total_orders_diff_%"] = cur[f"prev_{n}_total_orders_diff"] / cur[f"prev_{n}_total_orders"].replace(0,1)
    
    cur[f"prev_{n}_sell_price_main_diff"] = cur["sell_price_main"] - cur[f"prev_{n}_sell_price_main"]
    cur[f"prev_{n}_sell_price_main_diff_%"] = cur[f"prev_{n}_sell_price_main_diff"] / cur[f"prev_{n}_sell_price_main"].replace(0,1)
    
    cur[f"prev_{n}_Max_Discount_diff"] = cur["Max_Discount"] - cur[f"prev_{n}_Max_Discount"]
    cur[f"prev_{n}_Max_Discount_diff_%"] = cur[f"prev_{n}_Max_Discount_diff"] / cur[f"prev_{n}_Max_Discount"].replace(0,1)
    
    cur[f"prev_{n}_single_order_value_diff"] = (cur["single_order_value"] - cur[f"prev_{n}_single_order_value"]).astype("float32")
    cur[f"prev_{n}_single_order_value_diff_%"] = cur[f"prev_{n}_single_order_value_diff"] / cur[f"prev_{n}_single_order_value"].replace(0,1)

    
    cur[f"prev_{n}_warehouse"] = cur["warehouse"].shift(n)
    cur[f"prev_{n}_unique_id"] = cur["unique_id"].shift(n)
    
    cur[f"prev_{n}_flag"] = (
        (cur[f"prev_{n}_warehouse"] == cur["warehouse"]) * 
        (cur[f"prev_{n}_unique_id"] == cur["unique_id"])
        ).astype(int)
    
    
    useless = [f"prev_{n}_warehouse", f"prev_{n}_unique_id", f"prev_{n}_flag"]
    
    
    set_null_cols = [
        f"prev_{n}_total_orders_diff", f"prev_{n}_total_orders_diff_%", 
        f"prev_{n}_sell_price_main_diff", f"prev_{n}_sell_price_main_diff_%",
        f"prev_{n}_Max_Discount_diff", f"prev_{n}_Max_Discount_diff_%",
        f"prev_{n}_sales", f"prev_{n}_total_orders", 
        f"prev_{n}_sell_price_main", f"prev_{n}_Max_Discount", 
        f"prev_{n}_single_order_value", f"prev_{n}_single_order_value_diff",
        f"prev_{n}_single_order_value_diff_%"
        ]
    
    cur.loc[(cur[f"prev_{n}_flag"]==0), set_null_cols] = np.nan
    
    for col in useless:
        del cur[col] 
    
    ###############################################################################
    
    N_shifts = [2,3,4,5,6, 7,9,11, 14,21,28, 30,35]

    for n in range(2, max(N_shifts)+1):
        
        cur[f"prev_{n}_sales"] = cur["sales"].shift(n).astype("float32")
        cur[f"prev_{n}_total_orders"] = cur["total_orders"].shift(n).astype("float32")
        cur[f"prev_{n}_sell_price_main"] = cur["sell_price_main"].shift(n).astype("float32")
        cur[f"prev_{n}_Max_Discount"] = cur["Max_Discount"].shift(n).astype("float32")
        cur[f"prev_{n}_single_order_value"] = cur["single_order_value"].shift(n).astype("float32")



        # diff

        cur[f"prev_{n}_total_orders_diff"] = (cur["total_orders"] - cur[f"prev_{n}_total_orders"]).astype("float32")
        cur[f"prev_{n}_total_orders_diff_%"] = (cur[f"prev_{n}_total_orders_diff"] / cur[f"prev_{n}_total_orders"].replace(0,1)).astype("float32")
        
        cur[f"prev_{n}_sell_price_main_diff"] = (cur["sell_price_main"] - cur[f"prev_{n}_sell_price_main"]).astype("float32")
        cur[f"prev_{n}_sell_price_main_diff_%"] = (cur[f"prev_{n}_sell_price_main_diff"] / cur[f"prev_{n}_sell_price_main"].replace(0,1)).astype("float32")
        
        cur[f"prev_{n}_Max_Discount_diff"] = (cur["Max_Discount"] - cur[f"prev_{n}_Max_Discount"]).astype("float32")
        cur[f"prev_{n}_Max_Discount_diff_%"] = (cur[f"prev_{n}_Max_Discount_diff"] / cur[f"prev_{n}_Max_Discount"].replace(0,1)).astype("float32")
        
    # =============================================================================
        cur[f"prev_{n}_single_order_value_diff"] = (cur["single_order_value"] - cur[f"prev_{n}_single_order_value"]).astype("float32")
    #=============================================================================
        cur[f"prev_{n}_warehouse"] = cur["warehouse"].shift(n).astype("float32")
        cur[f"prev_{n}_unique_id"] = cur["unique_id"].shift(n).astype("float32")


    cols = [
        f"total_orders_diff", f"total_orders_diff_%", 
        f"sell_price_main_diff", f"sell_price_main_diff_%",
        f"Max_Discount_diff", f"Max_Discount_diff_%",
        f"sales", f"total_orders", 
        f"sell_price_main", f"Max_Discount",
        f"single_order_value", f"single_order_value_diff",
        ]


    ## 

    for col in cols:
        for c in N_shifts:
            cur_col_list = []
            
            sm_grt = 0
            sm_lt = 0
            sm_eq = 0
            
            c_col = f"prev_{c}_" + col
            for cn in range(1, c+1):
                cn_col = f"prev_{cn}_" + col
                cur_col_list.append(cn_col)
                
                if col != "sales":
                    sm_grt += cur[c_col] > cur[cn_col]
                    sm_lt += cur[c_col] < cur[cn_col]
                    sm_eq += cur[c_col] == cur[cn_col]
                
            
            cur[f"prev_{c}_"+ col+ "_Mean" ] = np.mean(cur[cur_col_list], axis = 1).astype("float32")
            cur[f"prev_{c}_"+ col+ "_Median" ] = np.median(cur[cur_col_list], axis = 1).astype("float32")
            cur[f"prev_{c}_"+ col+ "_Std" ] = np.std(cur[cur_col_list], axis = 1).astype("float32")
            cur[f"prev_{c}_"+ col+ "_Min" ] = np.min(cur[cur_col_list], axis = 1).astype("float32")
            cur[f"prev_{c}_"+ col+ "_Max" ] = np.max(cur[cur_col_list], axis = 1).astype("float32")
            
            
            # if (c>3) and  (col in ["sales", "total_orders"]):
            #     cur[f"prev_{c}_"+ col+ "_kurtosis" ] = kurtosis(cur[cur_col_list].fillna(0), axis = 1).astype("float32")
            #     cur[f"prev_{c}_"+ col+ "_skew" ] = skew(cur[cur_col_list].fillna(0), axis = 1).astype("float32")
            #     cur[f"prev_{c}_"+ col+ "_IQR" ] = iqr(cur[cur_col_list].fillna(0), axis = 1).astype("float32")
            #     cur[f"prev_{c}_"+ col+ "_65" ] = np.percentile(cur[cur_col_list], 65, axis = 1).astype("float32")
            
            
           
            if col != "sales":
                cur[f"prev_{c}_"+ col+ "_Grt" ] = sm_grt.astype("float32")
                cur[f"prev_{c}_"+ col+ "_Grt_Norm" ] = (cur[f"prev_{c}_"+ col+ "_Grt" ] / c).astype("float32")
                cur[f"prev_{c}_"+ col+ "_Lt" ] = sm_lt.astype("float32")
                cur[f"prev_{c}_"+ col+ "_Lt_Norm" ] = (cur[f"prev_{c}_"+ col+ "_Lt" ] / c).astype("float32")
                # cur[f"prev_{c}_"+ col+ "_Eq" ] = sm_eq
                # cur[f"prev_{c}_"+ col+ "_Eq_Norm" ] = (cur[f"prev_{c}_"+ col+ "_Eq" ] / c).astype("float32")

                    

    ###############################################################################

    # Removing Unnecessary columns

    col_dict = {} 


    # Creating columns name list for each N, Used for setting values to NaN for this columns

    for n in range(2,36):
        nl = [
            f"prev_{n}_sales", f"prev_{n}_total_orders",
            f"prev_{n}_sell_price_main", f"prev_{n}_Max_Discount",
            f"prev_{n}_total_orders_diff", f"prev_{n}_total_orders_diff_%",
            f"prev_{n}_sell_price_main_diff", f"prev_{n}_sell_price_main_diff_%",
            f"prev_{n}_Max_Discount_diff", f"prev_{n}_Max_Discount_diff_%",
            f"prev_{n}_single_order_value", f"prev_{n}_single_order_value_diff",
            
        ]
        
        if n in N_shifts:
        
            for col in cols:
                nl  += [
                    f"prev_{n}_"+ col+ "_Mean" ,
                    f"prev_{n}_"+ col+ "_Median",
                    f"prev_{n}_"+ col+ "_Std",
                    f"prev_{n}_"+ col+ "_Min",
                    f"prev_{n}_"+ col+ "_Max", 
            
                ] 
                
                if col != "sales":
                    
                    nl += [
                        f"prev_{n}_"+ col+ "_Grt", 
                        f"prev_{n}_"+ col+ "_Grt_Norm",
                        f"prev_{n}_"+ col+ "_Lt",
                        f"prev_{n}_"+ col+ "_Lt_Norm",
                        # f"prev_{n}_"+ col+ "_Eq",
                        # f"prev_{n}_"+ col+ "_Eq_Norm"
                    ]
                
                
        temp = []
        for col in nl :
            if col in features: 
                temp.append(col)
        col_dict[n] = temp


    ### Setting values to NaN where flag is False

    shifts = [i for i in range(2,36)] 

       
    for t,n in enumerate(range(2,36)):
        cur[f"prev_{n}_flag"] = (
            (cur[f"prev_{n}_warehouse"] == cur["warehouse"]) * 
            (cur[f"prev_{n}_unique_id"] == cur["unique_id"])
            ).astype(int)
        
        for nn in shifts[t:]:
            cur.loc[(cur[f"prev_{n}_flag"]==0), col_dict[nn]] = np.nan

    
    
    import gc 
    
    del sm_grt, sm_lt, sm_eq
    gc.collect()
    
    for col in col_dict[2]: 
        if col not in features:
            if col in cur:
                del cur[col]

    print("Prev32 DONe")
    
    ###########################################################################
    ###########################################################################
    
    # for n in range(1, 36):
        
        # cur[f"prev_{n}_sales"] = cur["sales"].shift(n).astype("float32")
        # cur[f"prev_{n}_total_orders"] = cur["total_orders"].shift(n).astype("float32")
        # cur[f"prev_{n}_unique_id"] = cur["unique_id"].shift(n).astype("float32")


    window_dict = {
        3  : [2, 3, 4, 6, 7, 12, 20],
        5  : [2, 3, 5, 6, 7, 12, 18], 
        7  : [2, 4, 5, 7, 14, 21], 
        14 : [2, 5, 7, 14, 17, 21]
     }

    for col in ["sales", "total_orders"]:
        for w in [3, 5, 7, 14]:
            for st in window_dict[w]:
                cur_col_list = []
                
                for c in range(st, st + w):
                    cur_col_list.append(f"prev_{c}_" + col)
                    
                cur[f"prev_{st}-{st+w}_" + col + "_Sum" ] = np.sum(cur[cur_col_list], axis = 1).astype("float32")
                cur[f"prev_{st}-{st+w}_" + col + "_Mean" ] = np.mean(cur[cur_col_list], axis = 1).astype("float32")
                cur[f"prev_{st}-{st+w}_" + col + "_Median" ] = np.median(cur[cur_col_list], axis = 1).astype("float32")
                cur[f"prev_{st}-{st+w}_" + col + "_Std" ] = np.std(cur[cur_col_list], axis = 1).astype("float32")
                cur[f"prev_{st}-{st+w}_" + col + "_Min" ] = np.min(cur[cur_col_list], axis = 1).astype("float32")
                cur[f"prev_{st}-{st+w}_" + col + "_Max" ] = np.max(cur[cur_col_list], axis = 1).astype("float32")
                
                cur[f"prev_{st}-{st+w}_"+ col + "_Range" ] = ( 
                    cur[f"prev_{st}-{st+w}_"+ col + "_Max" ] - cur[f"prev_{st}-{st+w}_"+ col + "_Min"]
                    )
                
     
    set_null_dict = {}
    for st in [3, 5, 7, 14]:
        set_null_dict[st] = {}



    for col in ["sales", "total_orders"]:
        for w in [3, 5, 7, 14]:
            for st in window_dict[w]:
                
                nl = [ 
                        f"prev_{st}-{st+w}_"+ col + "_Mean" ,
                        f"prev_{st}-{st+w}_"+ col + "_Median" ,
                        f"prev_{st}-{st+w}_"+ col + "_Std" ,
                        f"prev_{st}-{st+w}_"+ col + "_Max" ,
                        f"prev_{st}-{st+w}_" + col + "_Sum",
                        f"prev_{st}-{st+w}_"+ col + "_Range"
                        
                        ]
                    
                
                if st in set_null_dict[w]: 
                    set_null_dict[w][st] += nl 
                else: 
                    set_null_dict[w][st] = nl


    #cur["prev_1_unique_id"] = cur["unique_id"].shift(1)

    for w in [3, 5, 7, 14]:
        ww = w-1
        
        for nn in set_null_dict[w].keys():
            
            temp = []
            
            for col in set_null_dict[w][nn]:
                if col in features: 
                    temp.append(col)

            flag = (
                cur[f"prev_{nn}_unique_id"] != cur["unique_id"]) | (
                cur[f"prev_{nn + ww}_unique_id"] != cur["unique_id"]
                )
            
            cur.loc[flag, temp] = np.nan 
        
    
    print("window DONe")
    ###########################################################################
    ###########################################################################
    N_shifts = [2,3,4,5,6, 7,9,11, 14,21,28, 30,35]

    for n in range(1, max(N_shifts)+1):
        cur[f"prev_{n}_sales"] = cur["sales"].shift(n).astype("float32")
        

    #cur["prod_name"] = names[0]
    grp = cur.groupby(["date","warehouse","prod_name"])


    agg = "mean"

    for agg in ["mean", "median", "std"]:
        
        agg_grp = grp["sales"].transform(agg)
        cur[f"PROD_{agg}"] = agg_grp

        for n in range(1, 36):
            cur[f"prev_{n}_PROD_{agg}"] = cur[f"PROD_{agg}"].shift(n).astype("float32")
        
        for n in N_shifts: 
            cur_col_list = []
            if n != 1:
                for _ in range(1, n+1):
                    cur_col_list.append(f"prev_{_}_PROD_{agg}")
                    
                cur[f"prev_{n}_PROD_{agg}_MEAN"] = np.mean(cur[cur_col_list], axis = 1).astype("float32")
                cur[f"prev_{n}_PROD_{agg}_MEDIAN"] = np.median(cur[cur_col_list], axis = 1).astype("float32")
                cur[f"prev_{n}_PROD_{agg}_STD"] = np.std(cur[cur_col_list], axis = 1).astype("float32")
                cur[f"prev_{n}_PROD_{agg}_MIN"] = np.min(cur[cur_col_list], axis = 1).astype("float32")
                cur[f"prev_{n}_PROD_{agg}_MAX"] = np.max(cur[cur_col_list], axis = 1).astype("float32")
                cur[f"prev_{n}_PROD_{agg}_RANGE"] = (cur[f"prev_{n}_PROD_{agg}_MAX"]- cur[f"prev_{n}_PROD_{agg}_MIN"]).astype("float32")
                
                
                cur[f"prev_{n}_prod_sales_{agg}_MEAN_diff"] = (cur[f"prev_{n}_PROD_{agg}_MEAN"] - cur[f"prev_{n}_sales"]).astype("float32")
                cur[f"prev_{n}_prod_sales_{agg}_MEAN_diff_%"] = (cur[f"prev_{n}_prod_sales_{agg}_MEAN_diff"] / cur[f"prev_{n}_PROD_{agg}_MEAN"]).astype("float32")
                
                cur[f"prev_{n}_prod_sales_{agg}_MEDIAN_diff"] = (cur[f"prev_{n}_PROD_{agg}_MEDIAN"] - cur[f"prev_{n}_sales"]).astype("float32")
                cur[f"prev_{n}_prod_sales_{agg}_MEDIAN_diff_%"] = (cur[f"prev_{n}_prod_sales_{agg}_MEDIAN_diff"] / cur[f"prev_{n}_PROD_{agg}_MEDIAN"]).astype("float32")
                
                
                if n > 3:
                    gtmean = 0 
                    gtmedian = 0
                    for col in cur_col_list:
                        gtmean += (cur[f"prev_{n}_PROD_{agg}_MEAN"] > cur[col]).astype(int)
                        gtmedian += (cur[f"prev_{n}_PROD_{agg}_MEDIAN"] > cur[col]).astype(int)
                    
                    cur[f"prev_{n}_PROD_{agg}_MEAN_GT"] = gtmean
                    cur[f"prev_{n}_PROD_{agg}_MEDIAN_GT"] = gtmedian
                    
        

    col_dict = {}

    for n in range(1, 36):
        nn = []
        for agg in ["mean", "median", "std"]:
            for _ in range(1, 36):
                if _>=n: 
                    nn+= [f"prev_{_}_PROD_{agg}"]
                    
            
            for _ in [1, 2, 3, 4, 5, 6, 7, 9, 11, 14, 21, 28, 30, 35]:
                if _>=n:
                    nn += [
                        f"prev_{_}_PROD_{agg}_MEAN", 
                        f"prev_{_}_PROD_{agg}_MEDIAN",
                        f"prev_{_}_PROD_{agg}_STD", 
                        f"prev_{_}_PROD_{agg}_MIN",
                      f"prev_{_}_PROD_{agg}_MAX", 
                      f"prev_{_}_PROD_{agg}_RANGE", 
                      f"prev_{_}_prod_sales_{agg}_MEAN_diff", 
                      f"prev_{_}_prod_sales_{agg}_MEAN_diff_%",
                      f"prev_{_}_prod_sales_{agg}_MEDIAN_diff",
                      f"prev_{_}_prod_sales_{agg}_MEDIAN_diff_%"
                      ]
            
                    if n>=3: 
                        
                        nn += [
                            f"prev_{_}_PROD_{agg}_MEAN_GT",
                            f"prev_{_}_PROD_{agg}_MEDIAN_GT"
                            ]
        temp = []
        for col in nn: 
            if col in features: 
                temp.append(col)
        col_dict[n] = temp
        

    for i in range(1, 36):

        cur.loc[cur["unique_id"] != cur["unique_id"].shift(i), col_dict[i]] = np.nan
        
        
    for col in col_dict[1]: 
        if col not in features:
            if col in cur:
                del cur[col]

    print("prod done")
    
    #######################################################
    
    
    N_shifts = [2, 3, 4, 5, 6, 7, 9, 11, 14, 21, 28, 30, 35]
    columns = [
        "type_0_discount",
        "type_1_discount",
        "type_2_discount",
        "type_3_discount",
        "type_4_discount", 
        "type_5_discount"
        ]

    for col in columns: 
        for n in range(1, 36):
            cur[f"prev_{n}_{col}"] = cur[col].shift(n).astype("float32")
            

    for col in columns: 
        for n in N_shifts: 
            cur_col_list = []
            
            for _ in range(1, n+1): 
                cur_col_list += [f"prev_{_}_{col}"]
                
            cur[f"prev_{n}_{col}_MEAN"] = np.mean(cur[cur_col_list], axis = 1).astype("float32")
            
            
    col_dict = {}

    for n in range(2, 36):
        nn = []
        for col in columns: 
            
            for _ in range(1, 36):
                if n <= _:
                    nn += [f"prev_{_}_{col}"]
                
                
            for _ in N_shifts: 
                
                if n <= _:
                    nn += [f"prev_{_}_{col}_MEAN"]
                    
        temp = []
        for col in nn: 
            if col in features: 
                temp.append(col)
                
        col_dict[n] = temp
                

    for i in range(2, 35): 

        cur.loc[cur["unique_id"]!= cur["unique_id"].shift(i), col_dict[i]] = np.nan
            

    for col in columns: 
        for n in range(1, 36):
            if n in N_shifts + [1]: 
                continue 
            else:
                del cur[f"prev_{n}_{col}"]
                
    
    for col in col_dict[2]: 
        if col not in features:
            if col in cur:
                del cur[col]
    
    
    #######################################################################
    
    

    cur["disc_cat"] = (cur["type_0_discount"] // .1) + 1
    cur.loc[cur["type_0_discount"]==0, "disc_cat"] = 0 

    val_str = cur["prod_name"] + "_" + cur["disc_cat"].astype(int).astype(str)


    for agg in ["mean", "median"]:
        
        cur[f"type0_Cat_{agg}"] = val_str.map(discount_dict[0][agg])
        

    for d in [1,2,3,4,5,6]:
        cur["disc_cat"] = cur[f"type_{d}_discount"]>0
        val_str = cur["prod_name"] + "_" + cur["disc_cat"].astype(str)
        
        for agg in ["mean", "median"]:
            cur[f"type{d}_cat_{agg}"] = val_str.map(discount_dict[d][agg])
        
        
        
                
                
    
    
        
        
    ###########################################################################
    
    
    N_shifts = [2, 3, 4, 5, 6, 7, 9, 11, 14, 21, 28, 30, 35]
    
    for n in range(1, max(N_shifts)+1):
        
        cur[f"prev_{n}_sales"] = cur["sales"].shift(n).astype("float32")
        
    
    
    col = "sales"
    
    for c in N_shifts:
        cur_col_list = []
        
        sm_grt = 0
        sm_lt = 0
        sm_eq = 0
        
        c_col = f"prev_{c}_" + col
        
        for cn in range(1, c+1):
            cn_col = f"prev_{cn}_" + col
            cur_col_list.append(cn_col)
            
    
            sm_grt += (cur[c_col] > cur[cn_col]).astype("int")
            sm_lt += (cur[c_col] < cur[cn_col]).astype("int")
            sm_eq += (cur[c_col] == cur[cn_col]).astype("int")
            
        
        # cur[f"prev_{c}_"+ col+ "_Mean" ] = np.mean(cur[cur_col_list], axis = 1).astype("float32")
        # cur[f"prev_{c}_"+ col+ "_Median" ] = np.median(cur[cur_col_list], axis = 1).astype("float32")
        # cur[f"prev_{c}_"+ col+ "_Std" ] = np.std(cur[cur_col_list], axis = 1).astype("float32")
        # cur[f"prev_{c}_"+ col+ "_Min" ] = np.min(cur[cur_col_list], axis = 1).astype("float32")
        # cur[f"prev_{c}_"+ col+ "_Max" ] = np.max(cur[cur_col_list], axis = 1).astype("float32")
        
        
        cur[f"prev_{c}_"+ col+ "_Range" ] = (cur[f"prev_{c}_"+ col+ "_Max" ] - cur[f"prev_{c}_"+ col+ "_Min" ]).astype("float32")
        
        cur[f"prev_{c}_"+ col+ "_Grt" ] = sm_grt.astype("float32")
        cur[f"prev_{c}_"+ col+ "_Grt_Norm" ] = (cur[f"prev_{c}_"+ col+ "_Grt" ] / c).astype("float32")
        cur[f"prev_{c}_"+ col+ "_Lt" ] = sm_lt.astype("float32")
        cur[f"prev_{c}_"+ col+ "_Lt_Norm" ] = (cur[f"prev_{c}_"+ col+ "_Lt" ] / c).astype("float32")
        
    
    for nn in [1, 2, 3, 4, 5, 6, 7, 9, 11, 14, 21, 28, 30, 35] :
        for n in N_shifts:
    
            
            if nn<n:
                # Mean Diff
                cur[f"prev_{nn}-{n}_sales_Mean_diff"] = (cur[f"prev_{n}_sales_Mean"] - cur[f"prev_{nn}_sales"]).astype("float32")
                cur[f"prev_{nn}-{n}_sales_Mean_diff_%"] = (cur[f"prev_{nn}-{n}_sales_Mean_diff"] / cur[f"prev_{n}_sales_Mean"]).astype("float32")
                
                # Median Diff
                cur[f"prev_{nn}-{n}_sales_Median_diff"] = (cur[f"prev_{n}_sales_Median"] - cur[f"prev_{nn}_sales"]).astype("float32")
                cur[f"prev_{nn}-{n}_sales_Median_diff_%"] = (cur[f"prev_{nn}-{n}_sales_Median_diff"] / cur[f"prev_{n}_sales_Median"]).astype("float32")
                
                # Range Diff
                cur[f"prev_{nn}-{n}_sales_Mean_diff_Range"] = (cur[f"prev_{nn}-{n}_sales_Mean_diff"] / cur[f"prev_{n}_"+ col+ "_Range" ]).astype("float32")
                cur[f"prev_{nn}-{n}_sales_Median_diff_Range"] = (cur[f"prev_{nn}-{n}_sales_Median_diff"] / cur[f"prev_{n}_"+ col+ "_Range" ]).astype("float32")
                
                # Std Diff
                cur[f"prev_{nn}-{n}_sales_Mean_diff_Std"] = (cur[f"prev_{nn}-{n}_sales_Mean_diff"] / cur[f"prev_{n}_"+ col+ "_Std" ]).astype("float32")
                cur[f"prev_{nn}-{n}_sales_Median_diff_Std"] = (cur[f"prev_{nn}-{n}_sales_Median_diff"] / cur[f"prev_{n}_"+ col+ "_Std" ]).astype("float32")
                
                # Max Diff
                cur[f"prev_{nn}-{n}_sales_Max_diff"] = (cur[f"prev_{n}_sales_Max"] - cur[f"prev_{nn}_sales"]).astype("float32")
                cur[f"prev_{nn}-{n}_sales_Max_diff_%"] = (cur[f"prev_{nn}-{n}_sales_Max_diff"] / cur[f"prev_{n}_sales_Max"]).astype("float32")
                
                # Min Diff
                cur[f"prev_{nn}-{n}_sales_Min_diff"] = (cur[f"prev_{nn}_sales"] - cur[f"prev_{n}_sales_Min"]).astype("float32")
                cur[f"prev_{nn}-{n}_sales_Min_diff_%"] = (cur[f"prev_{nn}-{n}_sales_Min_diff"] / cur[f"prev_{n}_sales_Min"]).astype("float32")
                
                
    
    col_dict = {}     
    col = "sales"
    
    for i in range(1, 36):
        lis = []
        
        for s in N_shifts: 
            
            if s>n: 
                lis += [
                    f"prev_{s}_"+ col+ "_Range",
                    f"prev_{s}_"+ col+ "_Grt", 
                    f"prev_{s}_"+ col+ "_Grt_Norm",
                    f"prev_{s}_"+ col+ "_Lt",
                    f"prev_{s}_"+ col+ "_Lt_Norm" 
                    ]
                
        for nn in [1, 2, 3, 4, 5, 6, 7, 9, 11, 14, 21, 28, 30, 35] :
            for n in N_shifts:
                
                if (nn>= i) or (n>=i):
                
                    lis += [    
                        f"prev_{nn}-{n}_sales_Mean_diff",
                        f"prev_{nn}-{n}_sales_Mean_diff_%",
                        f"prev_{nn}-{n}_sales_Median_diff",
                        f"prev_{nn}-{n}_sales_Median_diff_%",
                        f"prev_{nn}-{n}_sales_Mean_diff_Range",
                        f"prev_{nn}-{n}_sales_Median_diff_Range",
                        f"prev_{nn}-{n}_sales_Mean_diff_Std",
                        f"prev_{nn}-{n}_sales_Median_diff_Std",
                        f"prev_{nn}-{n}_sales_Max_diff",
                        f"prev_{nn}-{n}_sales_Max_diff_%",
                        f"prev_{nn}-{n}_sales_Min_diff",
                        f"prev_{nn}-{n}_sales_Min_diff_%"
                        ]
                        
        temp = []
        for col in lis: 
            if col in features: 
                temp.append(col)
        col_dict[i] = temp 
        
    
    for i in range(1, 36): 

        cur.loc[cur["unique_id"]!= cur["unique_id"].shift(i), col_dict[i]] = np.nan
        
    for col in col_dict[1]: 
        if col not in features:
            if col in cur:
                del cur[col]
        
    print("diff done")
    ###############################################################################
    
    cur["type_0_discount_Flag"] = cur["type_0_discount"]>0
    cur["type_0_discount_5_Flag"] = cur["type_0_discount"] >= 0.05
    cur["type_0_discount_10_Flag"] = cur["type_0_discount"] >= 0.1
    cur["type_0_discount_15_Flag"] = cur["type_0_discount"] >= 0.15
    cur["type_0_discount_20_Flag"] = cur["type_0_discount"] >= 0.2
    cur["type_0_discount_25_Flag"] = cur["type_0_discount"] >= 0.25
    
    cur["type_1_discount_Flag"] = cur["type_1_discount"]>0
    cur["type_2_discount_Flag"] = cur["type_2_discount"]>0
    cur["type_3_discount_Flag"] = cur["type_3_discount"]>0
    cur["type_4_discount_Flag"] = cur["type_4_discount"]>0
    cur["type_5_discount_Flag"] = cur["type_5_discount"]>0
    cur["type_6_discount_Flag"] = cur["type_6_discount"]>0
    
    
    cur["type_1_discount_10_Flag"] = (cur["type_1_discount"] > 0) & (cur["type_1_discount"] <= 0.1)
    cur["type_1_discount_20_Flag"] = (cur["type_1_discount"] > .1) & (cur["type_1_discount"] <= 0.2)
    cur["type_1_discount_21_Flag"] = cur["type_1_discount"] > .2
    
    cur["type_2_discount_10_Flag"] = (cur["type_2_discount"] > 0) & (cur["type_2_discount"] <= 0.1)
    cur["type_2_discount_20_Flag"] = (cur["type_2_discount"] > .1) & (cur["type_2_discount"] <= 0.2)
    cur["type_2_discount_21_Flag"] = cur["type_2_discount"] > .2
    
    cur["type_3_discount_10_Flag"] = (cur["type_3_discount"] > 0) & (cur["type_3_discount"] <= 0.1)
    cur["type_3_discount_20_Flag"] = (cur["type_3_discount"] > .1) & (cur["type_3_discount"] <= 0.2)
    cur["type_3_discount_21_Flag"] = cur["type_3_discount"] > .2
    
    cur["type_4_discount_10_Flag"] = (cur["type_4_discount"] > 0) & (cur["type_4_discount"] <= 0.1)
    cur["type_4_discount_20_Flag"] = (cur["type_4_discount"] > .1) & (cur["type_4_discount"] <= 0.2)
    cur["type_4_discount_21_Flag"] = cur["type_4_discount"] > .2
    
    cur["type_5_discount_10_Flag"] = (cur["type_5_discount"] > 0) & (cur["type_5_discount"] <= 0.1)
    cur["type_5_discount_20_Flag"] = (cur["type_5_discount"] > .1) & (cur["type_5_discount"] <= 0.2)
    cur["type_5_discount_21_Flag"] = cur["type_5_discount"] > .2
    
    
    cols = [
            "type_0_discount_Flag",
            "type_1_discount_Flag",
            "type_2_discount_Flag",
            "type_3_discount_Flag",
            "type_4_discount_Flag",
            "type_5_discount_Flag",
            "type_6_discount_Flag",
            
            "type_0_discount_5_Flag",
            "type_0_discount_15_Flag",
            "type_0_discount_25_Flag",
    
            "type_0_discount_10_Flag",
            "type_1_discount_10_Flag",
            "type_2_discount_10_Flag",
            "type_3_discount_10_Flag",
            "type_4_discount_10_Flag",
            "type_5_discount_10_Flag",
            
            "type_1_discount_20_Flag",
            "type_2_discount_20_Flag",
            "type_3_discount_20_Flag",
            "type_4_discount_20_Flag",
            "type_5_discount_20_Flag",
            "type_0_discount_20_Flag",
            
            "type_1_discount_21_Flag",
            "type_2_discount_21_Flag",
            "type_3_discount_21_Flag",
            "type_4_discount_21_Flag",
            "type_5_discount_21_Flag",
            ]
    
    
    for col in cols:
        
        cur[f"daily_{col}_count"] = cur.groupby(["warehouse", "date"])[col].transform("sum").astype("float32")
        cur[f"daily_{col}_mean"] = cur.groupby(["warehouse", "date"])[col].transform("mean").astype("float32")
    ###########################################################################
    
    
    cur["disc_flag"] = (cur["Max_Discount"]==0).astype(int)
    #cur = cur.copy()
    
    g_col =  ( 
        cur["warehouse"].astype(str) + \
            "__"+ cur["unique_id"].astype(str) + \
        "__"+ cur["disc_flag"].astype(str)
        )
        
       
    g_col2 =  ( 
        cur["warehouse"].astype(str) + \
            "__"+ cur["unique_id"].astype(str)
        )
        
        
    #sales_dict = cur.groupby(g_col)["sales"].agg(["max","mean", "median", "std"]).to_dict()
    
    #sell_price_dict = cur.groupby(g_col2)["sell_price_main"].agg(["max", "mean", "median", "std"]).to_dict()
    
    
    for agg in ["max", "mean", "median", "std"]:
        for col in ["sell_price_main"]:
            cur[col + "_OVERALL_" + agg] = g_col2.map(sell_price_dict[agg]).astype("float32")
                
            
    for agg in ["max","mean", "median", "std", "min", "range", "kurtosis", "skew", "iqr"]:
                
        cur["sales" + "_OVERALL_" + agg] = g_col.map(sales_dict[agg]).astype("float32")
    
    del cur["disc_flag"]
    del g_col, g_col2
    
    ###############################################################################
    
    N_shifts = [1, 2, 3, 5, 7, 10, 14, 21, 28, 35]


    for i in N_shifts:
        prev = cur["sales"].shift(i)
        for n in N_shifts:
            if i>=n:
                continue 

            cur[f"prev_{i}-{n}_sales_DIFF"] = cur["sales"].shift(n) - prev
            cur[f"prev_{i}-{n}_sales_DIFF_%"] = cur[f"prev_{i}-{n}_sales_DIFF"] / prev
            

    col_dict = {}


    for day in range(1, 36):
        nn = []
        for i in N_shifts:
            for n in N_shifts:
                if i>=n:
                    continue 
                
                if day<=n:
                    nn += [
                        f"prev_{i}-{n}_sales_DIFF", 
                        f"prev_{i}-{n}_sales_DIFF_%", 
                        ]
                
        col_dict[day] = nn.copy()


    for k in col_dict:
        
        for col in col_dict[k]:
            cur.loc[cur["unique_id"] != cur["unique_id"].shift(k), col] = np.nan
    
    
        
    ###############################################################################
    
    cur["PROD_COUNT"] = cur.groupby(["warehouse", "date", "prod_name"])["unique_id"].transform("nunique")
    cur["distint_PRODS"] = cur.groupby(["warehouse", "date"])["unique_id"].transform("nunique")

    cur["orders_per_prod"] = cur["total_orders"] / cur["distint_PRODS"]
    
    ###########################################################################
    
    

    del col_dict, N_shifts, names, cur_drop, cur_col_list, c_col,c,agg,  
    del v, useless, to_drop, shifts, s 
    del pos, ord_cols, neg, nn, nl, n
    
    
    
    ###############################################################################
    target = "sales"
    
    ttest = cur[cur["sample"]=="test"]
    
    del ttest[target]
    
    ###############################################################################
    splits = 5 
    UID = ttest["unique_id"].copy()

    lpreds = np.zeros((splits, len(ttest)))
    for idx in range(splits):
        model = pickle.load(open(f"/kaggle/input/lgb2-all/LGB2_709_NewParam/LGB2_709_NewParam_{idx}.pkl", "rb"))
        features = model.feature_name_
        print("MAE", len(features))
        lpreds[idx] = model.predict(ttest[features])



    lpreds2 = np.zeros((splits, len(ttest)))
    for idx in range(splits):
        model = pickle.load(open(f"/kaggle/input/lgb2-all/LGB2_MAPE/LGB2_mape_{idx}.pkl", "rb"))
        features = model.feature_name_
        print("MAPE",len(features))
        lpreds2[idx] = model.predict(ttest[features])


    lpreds3 = np.zeros((7, len(ttest)))
    for idx in range(7):
        model = pickle.load(open(f"/kaggle/input/lgb2-all/LGB2_495_7Folds/LGB2_495_7Folds_{idx}.pkl", "rb"))
        features = model.feature_name_
        print("7FOLDS",len(features))
        lpreds3[idx] = model.predict(ttest[features])




    model = pickle.load(open(f"/kaggle/input/lgb2-all/LGB2_BIGSINGLE/LGB2_BIGSINGLE.pkl", "rb"))
    features = model.feature_name_
    print("BIGSINGLE",len(features))
    lpreds4 = model.predict(ttest[features])

    
    lpreds5 = np.zeros((splits, len(ttest)))
    for idx in range(splits):
        model = pickle.load(open(f"/kaggle/input/lgb2-714-800-leaves/LGB2_714_{idx}.pkl", "rb"))
        features = model.feature_name_
        print("800 Leaves:", len(features))
        lpreds5[idx] = model.predict(ttest[features])


    lpreds6 = np.zeros((splits, len(ttest)))
    for idx in range(splits):
        model = pickle.load(open(f"/kaggle/input/lgb2-714gkf/LGB2_714GKF_{idx}.pkl", "rb"))
        features = model.feature_name_
        print("GKF", len(features))
        lpreds6[idx] = model.predict(ttest[features])


    hidden_units = [1000, 2000, 2000, 1000]
    dropout_rates = [0.10, 0.19, 0.27, 0.3, 0.23]

    for col in MLP_feats:
        if col == "preds": continue
        
        ttest[col] = ttest[col].fillna(-100)
        ttest[col] = ttest[col].replace(-np.inf, -100)
        ttest[col] = ttest[col].replace(np.inf, -100)

    for col in MLP_feats: 
        if col == "preds": continue
        sc = sc_dict[col]
        ttest[col] = sc.transform(ttest[col].values.reshape(-1,1)).reshape(-1)
        
    splits = 5
    mlpreds = np.zeros((splits, len(ttest)))

    for idx in range(5):
        model = create_mlp1(710, 1, hidden_units, dropout_rates)
        model.load_weights(f'/kaggle/input/lgb2-all/MLP_710_NewParam/MLP_LGB2_NewParams_{idx}.weights.h5')
        ttest["preds"] = lpreds[idx]
        sc = sc_dict["preds"]
        ttest["preds"] = sc.transform(ttest["preds"].values.reshape(-1,1)).reshape(-1)
        print(len(MLP_feats))
        mlpreds[idx] = model.predict(ttest[MLP_feats].values).reshape(-1)#.clip(0)


    lpreds = np.mean(lpreds, axis = 0)
    lpreds2 = np.mean(lpreds2, axis = 0)
    lpreds3 = np.mean(lpreds3, axis = 0)
    mlpreds = np.mean(mlpreds, axis = 0)
    lpreds5 = np.mean(lpreds5, axis = 0)
    lpreds6 = np.mean(lpreds6, axis = 0)
    
    preds = (
        (lpreds * 20) + 
        (lpreds2 * 24) + 
        (lpreds3 * 11) + 
        (lpreds4 * 8) + 
        (mlpreds * 16) + 
        (lpreds5 * 16) +
        (lpreds6 * 5)
    ) / 100

    temp = pd.DataFrame()
    
    temp["unique_id"] = UID
    temp["id"] = temp["unique_id"].astype(str) + "_" + ttest["date"].astype(str)
    temp["date"] = ttest["date"]
    temp["sales"] = preds.clip(0)
    
    final_preds = pd.concat((final_preds, temp[["id", "sales"]]))
    if cur_day == 1:
        print(final_preds)
    temp["sales"] = preds
    
    concat_df = pd.merge(
        test[test["date_num"]==cur_day],
        temp[["unique_id", "date", "sales"]],
        how = "inner", 
        on = ["unique_id", "date"]
        )
    concat_df["sample"] = "train"
    
    train = pd.concat((train, concat_df), axis = 0)
    train.sort_values(by = ["warehouse", "unique_id", "date"], inplace = True)
    
    
    for k,v in zip((temp["unique_id"].astype(str) + "_" + temp["date"].astype(str)), temp["sales"]):
        cor_val_dict[k] = v

        
    del temp, concat_df, k, v, idx, preds, ttest, cur, t, st, set_null_cols, set_null_dict 
    del lis, i, gtmean, gtmedian, discount_cols, columns, cols, col, cn, Ls, w, ww
        

    print(f"{cur_day} Predicted!")

1 done
2 done
3 done
4 done
5 done
6 done
7 done
8 done
9 done
Prev32 DONe
window DONe
prod done
diff done
MAE 709
MAE 709
MAE 709
MAE 709
MAE 709
MAPE 714
MAPE 714
MAPE 714
MAPE 714
MAPE 714
7FOLDS 495
7FOLDS 495
7FOLDS 495
7FOLDS 495
7FOLDS 495
7FOLDS 495
7FOLDS 495
BIGSINGLE 735
800 Leaves: 714
800 Leaves: 714
800 Leaves: 714
800 Leaves: 714
800 Leaves: 714
GKF 714
GKF 714
GKF 714
GKF 714
GKF 714
710
106/106 ━━━━━━━━━━━━━━━━━━━━ 4s 34ms/step
710
106/106 ━━━━━━━━━━━━━━━━━━━━ 4s 33ms/step
710
106/106 ━━━━━━━━━━━━━━━━━━━━ 4s 34ms/step
710
106/106 ━━━━━━━━━━━━━━━━━━━━ 4s 33ms/step
710
106/106 ━━━━━━━━━━━━━━━━━━━━ 4s 35ms/step
                     id       sales
42        12_2024-06-03   32.482888
85        20_2024-06-03   37.718722
128       25_2024-06-03   76.466020
213       41_2024-06-03   68.760583
256       46_2024-06-03  152.122718
...                 ...         ...
224049  5382_2024-06-03  210.018283
224176  5410_2024-06-03   22.674050
224219  5415_2024-06-03   45.346314
224296 

In [13]:
final_preds = pd.merge(final_preds, sub1, on = ["id"], how = "inner")
final_preds["sales_hat"] = (((final_preds["sales_x"] * 75) + (final_preds["sales_y"] * 25)) / 100).clip(0)

In [14]:
final_preds[["id", "sales_hat"]].to_csv("submission.csv", index = False)